# ⚡ FranchiseOps AI — Milestone 2
### Enterprise Multi-Agent Franchise Operations Platform


## Step 1 — Install Dependencies


In [1]:
!pip install kagglehub  -q streamlit pyngrok bcrypt pyjwt pandas numpy scikit-learn joblib transformers accelerate bitsandbytes plotly streamlit-option-menu faker kaggle


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.4/10.4 MB 50.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 11.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.9/40.9 MB 18.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 829.3/829.3 kB 21.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.1/2.1 MB 53.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.4/11.4 MB 90.0 MB/s eta 0:00:00


## Step 2 — Configure Secrets & Mount Google Drive


In [2]:
import os

def _get_secret(key):
    try:
        from google.colab import userdata
        val = userdata.get(key)
        if val: return val
    except Exception:
        pass
    return os.environ.get(key, "")

NGROK_AUTHTOKEN = _get_secret("NGROK_AUTHTOKEN")
HF_TOKEN        = _get_secret("HF_TOKEN")
KAGGLE_USERNAME = _get_secret("KAGGLE_USERNAME")
KAGGLE_KEY      = _get_secret("KAGGLE_KEY")
EMAIL_PASSWORD  = _get_secret("EMAIL_PASSWORD")
EMAIL_ID        = _get_secret("EMAIL_ID")
JWT_SECRET_KEY  = _get_secret("JWT_SECRET_KEY") or "franchiseops_ai-dev-secret"
ADMIN_EMAIL     = _get_secret("ADMIN_EMAIL_ID") or "infosys@ai"
ADMIN_PASSWORD  = _get_secret("ADMIN_PASSWORD") or "admin@123"

if KAGGLE_USERNAME: os.environ["KAGGLE_USERNAME"] = KAGGLE_USERNAME
if KAGGLE_KEY:      os.environ["KAGGLE_KEY"]      = KAGGLE_KEY

try:
    if os.path.exists("/content"):
        from google.colab import drive
        drive.mount("/content/drive", force_remount=False)
        STORAGE_DIR = "/content/drive/MyDrive/FranchiseOps_AI"
        print("✅ Google Drive mounted.")
    else:
        STORAGE_DIR = os.path.abspath("./data/FranchiseOps_AI")
except Exception as e:
    STORAGE_DIR = os.path.abspath("./data/FranchiseOps_AI")

os.makedirs(os.path.join(STORAGE_DIR, "models", "hf_cache"), exist_ok=True)
os.makedirs(os.path.join(STORAGE_DIR, "models", "kaggle_cache"), exist_ok=True)
print(f"📁 Storage: {STORAGE_DIR}")
print(f"🔑 HF_TOKEN: {'✅' if HF_TOKEN else '❌ set in Colab Secrets'}")
print(f"🔑 ngrok:    {'✅' if NGROK_AUTHTOKEN else '❌ set in Colab Secrets'}")


📁 Storage: /content/data/FranchiseOps_AI
🔑 HF_TOKEN: ✅
🔑 ngrok:    ✅


## Step 3 — Verify GPU & Load Qwen-2.5-3B (4-bit NF4)


In [3]:
import os

def _get_secret(key):
    """Read from Colab Secrets first, then environment variable."""
    try:
        from google.colab import userdata
        val = userdata.get(key)
        if val: return val
    except Exception:
        pass
    return os.environ.get(key, "")

# ── Load all 7 secrets (set these in Colab Secrets panel) ──────────────────
NGROK_AUTHTOKEN = _get_secret("NGROK_AUTHTOKEN")
HF_TOKEN        = _get_secret("HF_TOKEN")
KAGGLE_USERNAME = _get_secret("KAGGLE_USERNAME")
KAGGLE_KEY      = _get_secret("KAGGLE_KEY")
EMAIL_PASSWORD  = _get_secret("EMAIL_PASSWORD")
EMAIL_ID        = _get_secret("EMAIL_ID")
JWT_SECRET_KEY  = _get_secret("JWT_SECRET_KEY") or "franchiseops_ai-dev-secret"
ADMIN_EMAIL     = _get_secret("ADMIN_EMAIL_ID") or "infosys@ai"
ADMIN_PASSWORD  = _get_secret("ADMIN_PASSWORD") or "admin@123"

# Expose Kaggle credentials for the kaggle library
if KAGGLE_USERNAME: os.environ["KAGGLE_USERNAME"] = KAGGLE_USERNAME
if KAGGLE_KEY:      os.environ["KAGGLE_KEY"]      = KAGGLE_KEY

# ── Mount Google Drive (auto-detected in Colab) ─────────────────────────────
try:
    if os.path.exists("/content"):
        from google.colab import drive
        drive.mount("/content/drive", force_remount=False)
        STORAGE_DIR = "/content/drive/MyDrive/FranchiseOps_AI"
        print("✅ Google Drive mounted.")
    else:
        STORAGE_DIR = os.path.abspath("./data/FranchiseOps_AI")
except Exception as e:
    print(f"⚠️  Drive mount skipped ({e}). Using local storage.")
    STORAGE_DIR = os.path.abspath("./data/FranchiseOps_AI")

os.makedirs(STORAGE_DIR, exist_ok=True)
os.makedirs(os.path.join(STORAGE_DIR, "models"), exist_ok=True)
os.makedirs(os.path.join(STORAGE_DIR, "models", "kaggle_cache"), exist_ok=True)
os.makedirs(os.path.join(STORAGE_DIR, "models", "hf_cache"), exist_ok=True)

print(f"\n📁 Storage:  {STORAGE_DIR}")
print(f"🔑 JWT:      {'✅ from Colab Secrets' if _get_secret('JWT_SECRET_KEY') else '⚠️  using dev default'}")
print(f"🔑 Admin:    {ADMIN_EMAIL}")
print(f"🔑 HF_TOKEN: {'✅' if HF_TOKEN else '❌ set in Colab Secrets'}")
print(f"🔑 Kaggle:   {'✅' if KAGGLE_KEY else '❌ optional — synthetic fallback'}")
print(f"🔑 ngrok:    {'✅' if NGROK_AUTHTOKEN else '❌ set in Colab Secrets'}")
print(f"🔑 Email:    {'✅' if EMAIL_PASSWORD else '❌ optional'}")


⚠️  Drive mount skipped (Error: credential propagation was unsuccessful). Using local storage.

📁 Storage:  /content/data/FranchiseOps_AI
🔑 JWT:      ⚠️  using dev default
🔑 Admin:    infosys@ai
🔑 HF_TOKEN: ✅
🔑 Kaggle:   ✅
🔑 ngrok:    ✅
🔑 Email:    ✅


In [4]:
!nvidia-smi


Thu Jul 30 18:36:28 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   45C    P8             10W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [5]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

MODEL_ID = "Qwen/Qwen2.5-3B-Instruct"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID, quantization_config=bnb_config, device_map="auto",
)
print("✅ Qwen-2.5-3B loaded. Footprint (GB):", round(model.get_memory_footprint() / 1e9, 2))


config.json:   0%|          | 0.00/661 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

...

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

✅ Qwen-2.5-3B loaded. Footprint (GB): 2.01


## Step 4 — Write All Application Modules (`llm_engine`, `config`, `auth`, `db`, `agents`, `dashboard`)


In [7]:
%%writefile llm_engine_franchise.py
"""
llm_engine_franchise.py — FranchiseOps AI (Milestone 2)
Qwen-2.5-3B-Instruct (4-bit NF4) Copilot engine.
Adds synthesize_erp_action(): combines Agent 1/2/3 outputs into a structured
JSON ERP action, per Phase 3 of the Milestone 2 architecture spec.
"""
import os, json, re, torch, threading
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from config import HF_TOKEN

MODEL_ID  = "Qwen/Qwen2.5-3B-Instruct"
CACHE_DIR = "/content/drive/MyDrive/FranchiseOps_AI/models/hf_cache"
os.makedirs(CACHE_DIR, exist_ok=True)

_model     = None
_tokenizer = None
_load_lock = threading.Lock()


def get_model():
    global _model, _tokenizer
    if _model is not None:
        return _model, _tokenizer
    with _load_lock:
        if _model is not None:
            return _model, _tokenizer
        bnb_config = BitsAndBytesConfig(
            load_in_4bit=True,
            bnb_4bit_compute_dtype=torch.float16,
            bnb_4bit_quant_type="nf4",
            bnb_4bit_use_double_quant=True,
        )
        _tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, token=(HF_TOKEN or None), cache_dir=CACHE_DIR)
        try:
            _model = AutoModelForCausalLM.from_pretrained(
                MODEL_ID, quantization_config=bnb_config, device_map="auto",
                token=(HF_TOKEN or None), cache_dir=CACHE_DIR, low_cpu_mem_usage=True,
                attn_implementation="sdpa",
            )
        except Exception:
            _model = AutoModelForCausalLM.from_pretrained(
                MODEL_ID, quantization_config=bnb_config, device_map="auto",
                token=(HF_TOKEN or None), cache_dir=CACHE_DIR, low_cpu_mem_usage=True,
            )
        return _model, _tokenizer


def warmup_llm():
    try:
        get_model()
        print("✅ LLM warm-loaded.")
    except Exception as e:
        print(f"⚠️ LLM warmup failed: {e}")


def is_llm_loaded():
    return _model is not None


_warmup_thread_started = False


def start_background_warmup():
    global _warmup_thread_started
    if _warmup_thread_started:
        return
    _warmup_thread_started = True
    threading.Thread(target=warmup_llm, daemon=True).start()


def _run(msgs, max_tokens=100, greedy=True):
    """Returns None on any failure (GPU/model/token/etc.) instead of raising —
    callers fall back to rule-based text per Section 8's explicit requirement
    that Copilot must degrade gracefully, never crash, when the LLM is unavailable."""
    try:
        model, tok = get_model()
        tmpl   = tok.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)
        inputs = tok(tmpl, return_tensors="pt").to(model.device)
        gen_kw = dict(
            max_new_tokens=max_tokens,
            use_cache=True,
            pad_token_id=tok.eos_token_id,
            eos_token_id=tok.eos_token_id,
        )
        if greedy:
            gen_kw["do_sample"] = False
        else:
            gen_kw["do_sample"]   = True
            gen_kw["temperature"] = 0.2
            gen_kw["top_p"]       = 0.9
        with torch.inference_mode():
            out = model.generate(**inputs, **gen_kw)
        return tok.decode(out[0][inputs.input_ids.shape[1]:], skip_special_tokens=True).strip()
    except Exception as e:
        print(f"⚠️ LLM inference unavailable ({e}) — using rule-based fallback.")
        return None


def _repair_json(text):
    text = re.sub(r'```json\s*|\s*```', '', text)
    m = re.search(r"\{.*\}", text, re.DOTALL)
    if m:
        text = m.group(0)
    text = re.sub(r'(["]|\d|true|false)\s*\n\s*(["\w]+":)', r'\1,\n\2', text)
    text = re.sub(r'(["]|\d|true|false)\s+(["\w]+":)', r'\1, \2', text)
    text = re.sub(r',\s*\}', '}', text)
    return text


def generate_json(prompt, schema_keys=None):
    sys_p = "You are an AI franchise intelligence engine. Respond ONLY with a valid JSON object."
    if schema_keys:
        sys_p += f" Required keys: {', '.join(schema_keys)}."
    raw = _run(
        [{"role": "system", "content": sys_p}, {"role": "user", "content": prompt}],
        max_tokens=150, greedy=True,
    )
    if raw is None:
        return {"error": "LLM unavailable"}
    try:
        return json.loads(_repair_json(raw))
    except Exception:
        if schema_keys:
            out = {}
            for k in schema_keys:
                km = re.search(rf'"{k}"\s*:\s*"([^"]*)"|"{k}"\s*:\s*([^,\}}]+)', raw)
                if km:
                    out[k] = (km.group(1) if km.group(1) is not None else km.group(2)).strip()
                else:
                    out[k] = "N/A"
            if any(v != "N/A" for v in out.values()):
                return out
        return {"error": "JSON parse failed", "raw": raw}


AGENT_ROLES = {
    "agent1": ("Workforce Retention Agent",
               "You specialise in employee satisfaction, overtime fatigue, and attrition risk."),
    "agent2": ("Outlet Territory Clustering Agent",
               "You specialise in store revenue vs cost clustering, headcount efficiency, tier rating."),
    "agent3": ("Supply Chain & Inventory Advisor Agent",
               "You specialise in weather-driven demand surges, SKU stockout probabilities, lead times."),
}


def generate_debate_and_synthesis(user_query, agent1_context, agent2_context, agent3_context, db_stats=None):
    system_prompt = (
        "You are the FranchiseOps AI Multi-Agent Engine. "
        "Analyze the query and all data. Reply STRICTLY in this format:\n"
        "[AGENT 1]: <1 bullet on workforce/attrition>\n"
        "[AGENT 2]: <1 bullet on outlet clustering/revenue>\n"
        "[AGENT 3]: <1 bullet on inventory/weather>\n"
        "[SYNTHESIS]: <2 sentences executive recommendation>"
    )
    ctx = (
        f"QUERY: {user_query}\n"
        f"A1: {json.dumps(agent1_context)}\n"
        f"A2: {json.dumps(agent2_context)}\n"
        f"A3: {json.dumps(agent3_context)}"
    )
    if db_stats:
        ctx += f"\nDB: {json.dumps(db_stats)}"

    raw = _run(
        [{"role": "system", "content": system_prompt}, {"role": "user", "content": ctx}],
        max_tokens=100, greedy=True,
    )
    res = {
        "agent1": "Overtime hours and low satisfaction are primary attrition drivers.",
        "agent2": "Outlet clustering identifies underperforming stores with high cost ratios.",
        "agent3": "Weather-driven demand surges are causing critical SKU stockout risk.",
        "synthesis": raw if raw is not None else
            "⚠️ LLM unavailable — showing rule-based summary. Review overtime, revenue-tier, "
            "and stockout signals together before acting; retry once the model is loaded.",
    }
    if raw is None:
        return res
    try:
        for key, tag, nxt in [
            ("agent1", "AGENT 1", "AGENT 2"),
            ("agent2", "AGENT 2", "AGENT 3"),
            ("agent3", "AGENT 3", "SYNTHESIS"),
        ]:
            m = re.search(rf"\[{tag}\]:\s*(.*?)(?=\[{nxt}\]|\Z)", raw, re.DOTALL | re.IGNORECASE)
            if m:
                res[key] = m.group(1).strip()
        m = re.search(r"\[SYNTHESIS\]:\s*(.*)", raw, re.DOTALL | re.IGNORECASE)
        if m:
            res["synthesis"] = m.group(1).strip()
    except Exception:
        pass
    return res


def orchestrate_3_agents_query(user_question, agent1_context, agent2_context, agent3_context, db_stats=None):
    sys_p = (
        "You are FranchiseOps AI Orchestrator. "
        "Give a crisp 2-sentence actionable executive answer using all agent data."
    )
    ctx = (
        f"QUERY: {user_question}\n"
        f"A1: {json.dumps(agent1_context)}\n"
        f"A2: {json.dumps(agent2_context)}\n"
        f"A3: {json.dumps(agent3_context)}"
    )
    if db_stats:
        ctx += f"\nDB: {json.dumps(db_stats)}"
    raw = _run(
        [{"role": "system", "content": sys_p}, {"role": "user", "content": ctx}],
        max_tokens=90, greedy=True,
    )
    if raw is not None:
        return raw
    # Section 8 — rule-based fallback instead of a crash when GPU/LLM is unavailable
    a1 = agent1_context or {}
    a2 = agent2_context or {}
    a3 = agent3_context or {}
    return (
        f"⚠️ LLM unavailable (rule-based fallback). Based on current data: "
        f"attrition risk is {'elevated' if a1.get('attrition_risk', 0) > 0.5 else 'moderate'}, "
        f"outlet performance is {'strong' if a2.get('top_outlet') else 'mixed'}, "
        f"and inventory stockout risk is {'high' if a3.get('stockout_risk', 0) > 0.5 else 'manageable'}. "
        f"Recommend reviewing overtime hours, outlet tier standings, and reorder thresholds together."
    )


# ──────────────────────────────────────────────────────────────────────────
# Phase 3 — Generative Advisory: structured JSON ERP action
# ──────────────────────────────────────────────────────────────────────────
ERP_SCHEMA_KEYS = ["action_type", "target_outlet", "priority", "recommended_step", "expected_impact"]


def synthesize_erp_action(user_question, agent1_context, agent2_context, agent3_context, db_stats=None):
    """
    Synthesizes the 3 agents' numerical outputs into ONE structured JSON ERP action,
    e.g. {"action_type": "Staffing Intervention", "target_outlet": "OUT-105",
          "priority": "High", "recommended_step": "...", "expected_impact": "..."}
    Falls back to a rule-based JSON if the LLM output can't be parsed (never crashes).
    """
    prompt = (
        f"Given this franchise data, output ONE JSON ERP action object with keys "
        f"{ERP_SCHEMA_KEYS}.\n"
        f"USER REQUEST: {user_question}\n"
        f"AGENT 1 (Workforce): {json.dumps(agent1_context)}\n"
        f"AGENT 2 (Outlets/Revenue): {json.dumps(agent2_context)}\n"
        f"AGENT 3 (Inventory/Weather): {json.dumps(agent3_context)}\n"
        f"DB STATS: {json.dumps(db_stats or {})}"
    )
    result = generate_json(prompt, schema_keys=ERP_SCHEMA_KEYS)

    if "error" in result:
        # Rule-based fallback so the ERP Copilot page never crashes without a GPU/LLM
        result = {
            "action_type": "Retention Review",
            "target_outlet": (agent2_context or {}).get("top_outlet", "OUT-101"),
            "priority": "High" if (agent1_context or {}).get("attrition_risk", 0) > 0.5 else "Medium",
            "recommended_step": "Schedule a manager check-in for outlets flagged with high overtime "
                                 "and low satisfaction; review reorder thresholds for SKUs at stockout risk.",
            "expected_impact": "Rule-based fallback estimate (LLM unavailable): projected 5-10% reduction "
                                "in attrition risk and stockout incidents.",
            "note": "generated via rule-based fallback, not the LLM",
        }
    return result


Overwriting llm_engine_franchise.py


In [9]:
%%writefile config.py
"""
config.py — FranchiseOps AI (v3 FINAL)
All secrets from Colab userdata. KMEANS_MODEL_PATH = kmeans_outlets.joblib (spec compliant).
"""
import os

def _get_secret(key):
    try:
        from google.colab import userdata
        val = userdata.get(key)
        if val: return val
    except Exception:
        pass
    return os.environ.get(key, "")

try:
    from __main__ import (STORAGE_DIR, NGROK_AUTHTOKEN, HF_TOKEN,
                          KAGGLE_USERNAME, KAGGLE_KEY, EMAIL_PASSWORD,
                          ADMIN_EMAIL, ADMIN_PASSWORD, EMAIL_ID)
except ImportError:
    STORAGE_DIR    = ("/content/drive/MyDrive/FranchiseOps_AI"
                      if os.path.exists("/content/drive/MyDrive") else
                      os.path.abspath("./data/FranchiseOps_AI"))
    NGROK_AUTHTOKEN = _get_secret("NGROK_AUTHTOKEN")
    NGROK_AUTH_TOKEN = NGROK_AUTHTOKEN # Alias for launch cell compatibility
    HF_TOKEN        = _get_secret("HF_TOKEN")
    KAGGLE_USERNAME = _get_secret("KAGGLE_USERNAME")
    KAGGLE_KEY      = _get_secret("KAGGLE_KEY")
    EMAIL_PASSWORD  = _get_secret("EMAIL_PASSWORD")
    EMAIL_ID        = _get_secret("EMAIL_ID")
    JWT_SECRET_KEY  = _get_secret("JWT_SECRET_KEY") or "franchiseops-dev-secret-changeme"
    ADMIN_EMAIL     = _get_secret("ADMIN_EMAIL_ID")  or "infosys@ai"
    ADMIN_PASSWORD  = _get_secret("ADMIN_PASSWORD")  or "admin@123"

os.makedirs(STORAGE_DIR, exist_ok=True)
DB_PATH          = os.path.join(STORAGE_DIR, "franchiseops.db")
MODELS_DIR       = os.path.join(STORAGE_DIR, "models")
KAGGLE_CACHE_DIR = os.path.join(MODELS_DIR, "kaggle_cache")
os.makedirs(MODELS_DIR, exist_ok=True)
os.makedirs(KAGGLE_CACHE_DIR, exist_ok=True)

# Model paths (filenames match Infosys spec exactly)
AGENT1_MODEL_PATH = os.path.join(MODELS_DIR, "attrition_lr.joblib")
KMEANS_MODEL_PATH = os.path.join(MODELS_DIR, "kmeans_outlets.joblib")   # spec: kmeans_outlets
AGENT2_MODEL_PATH = KMEANS_MODEL_PATH                                    # alias
AGENT2_REG_PATH   = os.path.join(MODELS_DIR, "revenue_rf.joblib")
AGENT3_MODEL_PATH = os.path.join(MODELS_DIR, "inventory_demand_gb.joblib")


Overwriting config.py


In [11]:
%%writefile ui_theme.py
"""
ui_theme.py — FranchiseOps AI (Milestone 2)
Dark-mode enterprise dashboard theme.
Same COLORS keys, CSS class names (.pn-card, .pn-badge, .agent-badge) and
function signatures (inject_css, apply_theme, render_header, render_card,
risk_badge) as before — so admin_dash.py / agent2_franchise.py /
agent3_franchise.py / app.py need ZERO changes to pick this up.
"""
import streamlit as st

COLORS = {
    "bg_main":       "#0b0e14",
    "bg_card":       "#131720",
    "bg_alt":        "#1a1f2b",
    "text_heading":  "#f1f5f9",
    "text_body":     "#cbd5e1",
    "text_main":     "#cbd5e1",
    "text_muted":    "#7b8794",
    "border":        "#232a3a",
    "accent":        "#6366f1",
    "accent_subtle": "#818cf8",
    "accent_text":   "#ffffff",
    "cyan":          "#22d3ee",
    "pink":          "#f472b6",
    "green":         "#34d399",
    "yellow":        "#fbbf24",
    "red":           "#f87171",
}

DARK_DASHBOARD_CSS = f"""
<style>
@import url('https://fonts.googleapis.com/css2?family=Inter:wght@400;500;600;700;800&family=Space+Grotesk:wght@600;700&family=JetBrains+Mono:wght@500;700&display=swap');

html, body, [class*="css"], .stApp, [data-testid="stAppViewContainer"] {{
    font-family: 'Inter', sans-serif;
    color: {COLORS["text_body"]};
    background-color: {COLORS["bg_main"]} !important;
}}
[data-testid="stHeader"] {{
    background: transparent !important;
}}
[data-testid="stSidebar"] {{
    background-color: {COLORS["bg_alt"]} !important;
    border-right: 1px solid {COLORS["border"]};
}}

h1, h2, h3, h4, h5, h6 {{
    font-family: 'Space Grotesk', sans-serif;
    color: {COLORS["text_heading"]};
    font-weight: 700;
}}
p, span, label, div {{
    color: {COLORS["text_body"]};
}}

.pn-card {{
    background: {COLORS["bg_card"]};
    border: 1px solid {COLORS["border"]};
    border-radius: 14px;
    padding: 20px;
    margin-bottom: 20px;
    box-shadow: 0 4px 24px rgba(0,0,0,0.35);
    transition: border-color 0.15s ease, box-shadow 0.15s ease;
}}
.pn-card:hover {{
    border-color: {COLORS["accent"]};
    box-shadow: 0 6px 28px rgba(99,102,241,0.25);
}}
.pn-card-alt {{
    background: linear-gradient(145deg, {COLORS["bg_card"]}, {COLORS["bg_alt"]});
    border: 1px solid {COLORS["accent"]};
    border-radius: 14px;
    padding: 20px;
    margin-bottom: 20px;
    box-shadow: 0 4px 24px rgba(0,0,0,0.35);
}}

.pn-badge {{
    display: inline-block;
    padding: 4px 12px;
    border-radius: 999px;
    font-family: 'JetBrains Mono', monospace;
    font-weight: 700;
    font-size: 12px;
    color: #0b0e14;
    text-transform: uppercase;
    letter-spacing: 0.3px;
}}
.agent-badge {{
    display: inline-block;
    padding: 4px 14px;
    background: {COLORS["accent"]};
    color: {COLORS["accent_text"]};
    border-radius: 999px;
    font-family: 'Space Grotesk', sans-serif;
    font-weight: 700;
    font-size: 13px;
    box-shadow: 0 2px 12px rgba(99,102,241,0.45);
}}

/* Buttons */
div.stButton > button {{
    background: linear-gradient(135deg, {COLORS["accent"]}, {COLORS["accent_subtle"]}) !important;
    color: #ffffff !important;
    font-family: 'Space Grotesk', sans-serif !important;
    font-weight: 700 !important;
    border: none !important;
    border-radius: 10px !important;
    padding: 10px 22px !important;
    box-shadow: 0 4px 16px rgba(99,102,241,0.35) !important;
    transition: all 0.15s ease !important;
}}
div.stButton > button:hover {{
    transform: translateY(-1px) !important;
    box-shadow: 0 6px 22px rgba(99,102,241,0.55) !important;
}}

/* Inputs & selects */
div[data-baseweb="input"] > div, div[data-baseweb="select"] > div, .stTextInput input {{
    background: {COLORS["bg_alt"]} !important;
    border: 1px solid {COLORS["border"]} !important;
    border-radius: 8px !important;
    color: {COLORS["text_heading"]} !important;
}}
div[data-baseweb="input"] > div:focus-within, div[data-baseweb="select"] > div:focus-within {{
    border-color: {COLORS["accent"]} !important;
    box-shadow: 0 0 0 2px rgba(99,102,241,0.35) !important;
}}

/* Tabs */
button[data-baseweb="tab"] {{
    font-family: 'Space Grotesk', sans-serif !important;
    font-weight: 600 !important;
    color: {COLORS["text_muted"]} !important;
}}
button[data-baseweb="tab"][aria-selected="true"] {{
    color: {COLORS["text_heading"]} !important;
    border-bottom: 2px solid {COLORS["accent"]} !important;
}}

/* Dataframes / tables */
[data-testid="stDataFrame"] {{
    border: 1px solid {COLORS["border"]};
    border-radius: 10px;
}}

/* Metrics */
[data-testid="stMetric"] {{
    background: {COLORS["bg_card"]};
    border: 1px solid {COLORS["border"]};
    border-radius: 12px;
    padding: 12px 16px;
}}
[data-testid="stMetricValue"] {{
    color: {COLORS["text_heading"]} !important;
}}
</style>
"""


def inject_css():
    st.markdown(DARK_DASHBOARD_CSS, unsafe_allow_html=True)


def apply_theme():
    inject_css()


def render_header(title, subtitle="", icon="⚡"):
    inject_css()
    st.markdown(f"""
    <div style="background:{COLORS['bg_card']};border:1px solid {COLORS['border']};border-radius:16px;padding:22px 28px;margin-bottom:24px;box-shadow:0 4px 24px rgba(0,0,0,0.35);">
        <div style="display:flex;align-items:center;gap:16px;">
            <div style="font-size:42px;line-height:1;">{icon}</div>
            <div>
                <h1 style="margin:0;font-size:26px;letter-spacing:-0.5px;">{title}</h1>
                <p style="margin:4px 0 0;color:{COLORS['text_muted']};font-size:14px;">{subtitle}</p>
            </div>
        </div>
    </div>
    """, unsafe_allow_html=True)


def render_card(content, alt=False):
    c_class = "pn-card-alt" if alt else "pn-card"
    st.markdown(f'<div class="{c_class}">{content}</div>', unsafe_allow_html=True)


def risk_badge(text, level="Low"):
    color_map = {"Low": COLORS["green"], "Medium": COLORS["yellow"],
                 "High": COLORS["red"], "Critical": COLORS["red"]}
    c = color_map.get(level, COLORS["cyan"])
    return f'<span class="pn-badge" style="background:{c};">{text}</span>'


Overwriting ui_theme.py


In [13]:
%%writefile auth.py
"""
FranchiseOps AI - auth.py  (Milestone 2)
Login + progressive lockout (Section 5), Register + password strength (Section 6),
Forgot Password via Gmail OTP + resend cooldown (Section 5.1).
"""
import sqlite3, jwt, bcrypt, datetime, random, smtplib, ssl
from email.mime.text import MIMEText
import streamlit as st

try:
    from config import DB_PATH, JWT_SECRET_KEY, EMAIL_ID, EMAIL_PASSWORD
    JWT_SECRET = JWT_SECRET_KEY
except (ImportError, AttributeError):
    from config import DB_PATH
    JWT_SECRET = "super-secret-franchiseops-key-2026"
    EMAIL_ID, EMAIL_PASSWORD = "", ""

from ui_theme import COLORS
from db import (get_conn, get_user_lock_state, register_failed_login,
                 reset_lockout_on_success, get_otp_state, create_or_update_otp,
                 consume_otp)


# ──────────────────────────────────────────────────────────────────────────
# Low-level helpers
# ──────────────────────────────────────────────────────────────────────────
def hash_txt(t):
    return bcrypt.hashpw(t.encode(), bcrypt.gensalt()).decode()


def check_txt(t, h):
    try:
        return bcrypt.checkpw(t.encode(), h.encode()) if h else False
    except Exception:
        return False


def make_jwt(email, username, role="Franchise Owner"):
    return jwt.encode(
        {"email": email, "username": username, "role": role,
         "exp": datetime.datetime.utcnow() + datetime.timedelta(hours=6)},
        JWT_SECRET, algorithm="HS256")


def verify_jwt(token):
    try:
        return jwt.decode(token, JWT_SECRET, algorithms=["HS256"])
    except Exception:
        return None


@st.cache_resource
def init_auth():
    from db import init_db
    init_db()
    with get_conn() as conn:
        if not conn.execute("SELECT id FROM users WHERE email='infosys@ai'").fetchone():
            conn.execute("""INSERT OR IGNORE INTO users
                         (username, email, password_hash, security_question, security_answer_hash,
                          role, failed_attempts, lock_until, account_status)
                         VALUES (?, ?, ?, ?, ?, ?, 0, NULL, 'active')""",
                         ("Administrator", "infosys@ai", hash_txt("admin@123"),
                          "What is your pet name?", hash_txt("admin"), "Admin"))
            conn.commit()


# ──────────────────────────────────────────────────────────────────────────
# Section 6 — Password strength policy
# ──────────────────────────────────────────────────────────────────────────
def password_strength(pw: str):
    """Returns (badge, allowed, message)."""
    n = len(pw or "")
    if n < 5:
        return ("🔴 Weak", False,
                "Password too weak (minimum 5 characters required).")
    if n < 10:
        return ("🟡 Average", True,
                "🟡 Average strength (10+ characters recommended for enterprise security).")
    return ("🟢 Good", True, "🟢 Good password strength — proceed with bcrypt hashing.")


# ──────────────────────────────────────────────────────────────────────────
# Section 5.1 — Gmail OTP send + resend cooldown ladder
# ──────────────────────────────────────────────────────────────────────────
_COOLDOWN_LADDER = {0: 60, 1: 180, 2: 300}   # 1st, 2nd, 3rd resend
_COOLDOWN_MAX = 3600                          # 4th+


def _cooldown_for(resend_count):
    return _COOLDOWN_LADDER.get(resend_count, _COOLDOWN_MAX)


def _send_email(to_email, subject, body):
    """Real Gmail SMTP send if EMAIL_ID/EMAIL_PASSWORD are configured; console fallback otherwise.
    Also writes to /content/otp_debug.log so this is checkable regardless of how
    Streamlit's stdout is (or isn't) captured by the launching cell."""
    log_path = "/content/otp_debug.log"

    def _log(line):
        try:
            with open(log_path, "a") as f:
                f.write(f"{datetime.datetime.utcnow().isoformat()} | {line}\n")
        except Exception:
            pass

    if EMAIL_ID and EMAIL_PASSWORD:
        try:
            msg = MIMEText(body)
            msg["Subject"] = subject
            msg["From"] = EMAIL_ID
            msg["To"] = to_email
            ctx = ssl.create_default_context()
            with smtplib.SMTP("smtp.gmail.com", 587) as server:
                server.starttls(context=ctx)
                server.login(EMAIL_ID, EMAIL_PASSWORD)
                server.sendmail(EMAIL_ID, to_email, msg.as_string())
            _log(f"[SMTP SUCCESS] To: {to_email} | Subject: {subject}")
            return True
        except Exception as e:
            print(f"[EMAIL FALLBACK] SMTP send failed ({e}); printing to console instead.")
            _log(f"[SMTP FAILED] To: {to_email} | Error: {e}")
    print(f"[CONSOLE EMAIL] To: {to_email} | Subject: {subject}\n{body}")
    _log(f"[CONSOLE FALLBACK] To: {to_email} | Subject: {subject} | Body: {body}")
    return False


def request_otp(email):
    """Generates + sends an OTP, enforcing the resend cooldown ladder. Returns (ok, message)."""
    with get_conn() as conn:
        user = conn.execute("SELECT id FROM users WHERE email=?", (email,)).fetchone()
    if not user:
        return False, "Email not found."

    state = get_otp_state(email)
    now = datetime.datetime.utcnow()
    resend_count = 0
    if state:
        _id, resend_count, next_allowed = state
        if next_allowed:
            next_allowed_dt = datetime.datetime.fromisoformat(next_allowed)
            if now < next_allowed_dt:
                # Section 5.1 — exact per-attempt message, not a generic countdown
                if resend_count == 1:
                    return False, "⏳ Please wait 60 seconds before requesting another OTP."
                elif resend_count == 2:
                    return False, "⏳ Please wait 3 minutes before requesting another OTP."
                elif resend_count == 3:
                    return False, "⏳ Please wait 5 minutes before requesting another OTP."
                else:
                    return False, "⚠️ Too many OTP requests. Please wait 1 hour before trying again."

    otp = f"{random.randint(0, 999999):06d}"
    cooldown = _cooldown_for(resend_count)
    create_or_update_otp(email, hash_txt(otp), resend_count + 1, cooldown)
    _send_email(email, "FranchiseOps AI — Password Reset OTP",
                f"Your OTP is {otp}. It expires in 10 minutes. "
                f"If you didn't request this, ignore this email.")
    return True, "✅ OTP sent to your registered email."


def verify_otp_and_reset(email, otp_input, new_password):
    state = get_otp_state(email)
    if not state:
        return False, "No OTP request found. Please request a new OTP."
    otp_id, _resend, _next = state
    with get_conn() as conn:
        row = conn.execute("SELECT otp_hash, expires_at FROM otp_codes WHERE id=?", (otp_id,)).fetchone()
    if not row:
        return False, "OTP not found."
    otp_hash, expires_at = row
    if datetime.datetime.utcnow() > datetime.datetime.fromisoformat(expires_at):
        return False, "OTP has expired. Please request a new one."
    if not check_txt(otp_input, otp_hash):
        return False, "Incorrect OTP."

    badge, allowed, msg = password_strength(new_password)
    if not allowed:
        return False, msg

    with get_conn() as conn:
        conn.execute("UPDATE users SET password_hash=?, failed_attempts=0, lock_until=NULL, "
                     "account_status='active' WHERE email=?", (hash_txt(new_password), email))
        conn.commit()
    consume_otp(otp_id)
    return True, "Password reset successfully! Please sign in."


# ──────────────────────────────────────────────────────────────────────────
# UI
# ──────────────────────────────────────────────────────────────────────────
def render_auth_portal():
    init_auth()
    if "token" not in st.session_state:
        st.session_state["token"] = None

    st.markdown(f"""
    <div style="text-align:center;padding:1.5rem 0 1rem;">
        <div style="font-size:44px;margin-bottom:8px;">⚡</div>
        <h1 style="font-size:2rem !important;margin:0;">FranchiseOps AI Portal</h1>
        <p style="color:{COLORS['text_muted']};font-size:14px;margin:4px 0 0;">Enterprise Multi-Agent Franchise Intelligence System</p>
    </div>
    """, unsafe_allow_html=True)

    c1, c2, c3 = st.columns([1, 2, 1])
    with c2:
        tab1, tab2, tab3 = st.tabs(["🔐 Sign In", "📝 Register Account", "🔑 Reset Password (OTP)"])

        # ── TAB 1: Sign In (progressive lockout enforced) ────────────────────
        with tab1:
            login_email = st.text_input("Email / Username", key="l_email", placeholder="infosys@ai")
            login_pw = st.text_input("Password", type="password", key="l_pw", placeholder="••••••••")
            if st.button("🚀 Sign In to Portal", key="btn_login"):
                lock_state = get_user_lock_state(login_email)
                if not lock_state:
                    st.error("Invalid email/username or password.")
                else:
                    user_id, failed_attempts, lock_until, account_status = lock_state
                    now = datetime.datetime.utcnow()

                    if account_status == "locked":
                        st.error("❌ Account permanently locked due to 5 failed attempts. "
                                 "Only the System Administrator can unlock this account via the Admin Dashboard.")
                    elif lock_until and now < datetime.datetime.fromisoformat(lock_until):
                        remaining = int((datetime.datetime.fromisoformat(lock_until) - now).total_seconds())
                        mins = max(1, remaining // 60)
                        st.error(f"⏳ Account temporarily locked. Try again in ~{mins} minute(s).")
                    else:
                        with get_conn() as conn:
                            user = conn.execute(
                                "SELECT username, email, password_hash, role FROM users WHERE id=?",
                                (user_id,)).fetchone()
                        if user and check_txt(login_pw, user[2]):
                            reset_lockout_on_success(user_id)
                            st.session_state["token"] = make_jwt(user[1], user[0], user[3])
                            st.session_state["username"] = user[0]
                            st.session_state["role"] = user[3]
                            st.success(f"Welcome back, {user[0]} [{user[3]}]!")
                            st.rerun()
                        else:
                            msg = register_failed_login(user_id, failed_attempts)
                            if msg:
                                st.error(msg)
                            else:
                                st.error("Invalid email/username or password.")

        # ── TAB 2: Register (real-time password strength) ────────────────────
        with tab2:
            r_user = st.text_input("Username", key="r_u")
            r_email = st.text_input("Email Address", key="r_e")
            r_pw = st.text_input("Create Password", type="password", key="r_p")
            if r_pw:
                badge, allowed, msg = password_strength(r_pw)
                (st.success if allowed and badge == "🟢 Good" else
                 st.warning if allowed else st.error)(msg)
            r_role = st.selectbox("Select Enterprise Role",
                                   ["Franchise Owner", "Regional Operations Manager",
                                    "Store Manager", "Supply Chain Analyst"], key="r_role")
            r_q = st.selectbox("Security Question",
                                ["What is your pet name?", "What city were you born in?",
                                 "What is your favorite school teacher's name?"], key="r_q")
            r_a = st.text_input("Security Answer", key="r_a")
            if st.button("✨ Create Franchisee Account", key="btn_reg"):
                _badge, allowed, msg = password_strength(r_pw or "")
                if not (r_user and r_email and r_pw and r_a):
                    st.warning("Please fill out all fields.")
                elif not allowed:
                    st.warning(msg)
                else:
                    try:
                        with get_conn() as conn:
                            conn.execute(
                                "INSERT INTO users (username, email, password_hash, security_question, "
                                "security_answer_hash, role, failed_attempts, lock_until, account_status) "
                                "VALUES (?, ?, ?, ?, ?, ?, 0, NULL, 'active')",
                                (r_user, r_email, hash_txt(r_pw), r_q, hash_txt(r_a.lower().strip()), r_role))
                            conn.commit()
                        st.success(f"Account registered with role [{r_role}]! Please switch to Sign In tab.")
                    except Exception:
                        st.error("Registration failed: Email or username may already exist.")

        # ── TAB 3: Forgot Password via Gmail OTP + resend cooldown ───────────
        with tab3:
            f_email = st.text_input("Registered Email", key="f_e")
            colA, colB = st.columns(2)
            with colA:
                if st.button("📧 Send OTP", key="btn_send_otp"):
                    ok, msg = request_otp(f_email)
                    (st.success if ok else st.warning)(msg)
                    if ok:
                        st.session_state["otp_email"] = f_email
            with colB:
                if st.button("🔁 Resend OTP", key="btn_resend_otp"):
                    ok, msg = request_otp(f_email)
                    (st.success if ok else st.warning)(msg)

            if st.session_state.get("otp_email"):
                st.info(f"OTP sent to **{st.session_state['otp_email']}** (check inbox, or console log in dev mode).")
                otp_try = st.text_input("Enter OTP", key="f_otp")
                new_pw = st.text_input("New Password", type="password", key="f_npw")
                if new_pw:
                    _b, _a, msg = password_strength(new_pw)
                    st.caption(msg)
                if st.button("Confirm Password Reset", key="btn_f2"):
                    ok, msg = verify_otp_and_reset(st.session_state["otp_email"], otp_try, new_pw)
                    if ok:
                        st.success(msg)
                        st.session_state["otp_email"] = None
                    else:
                        st.error(msg)


Overwriting auth.py


In [15]:
%%writefile db.py
"""
db.py — FranchiseOps AI (Milestone 2)
Adds: progressive lockout columns, OTP tracking table, admin lifecycle helpers
(add_user / delete_user / unlock_user), on top of the Milestone 1 schema.
"""
import sqlite3
import datetime
from config import DB_PATH


def get_conn():
    return sqlite3.connect(DB_PATH, check_same_thread=False)


def _safe_alter(conn, sql):
    try:
        conn.execute(sql)
    except Exception:
        pass


def init_db():
    with get_conn() as conn:
        conn.execute("""CREATE TABLE IF NOT EXISTS outlets (
            outlet_id TEXT PRIMARY KEY, outlet_name TEXT, city TEXT,
            monthly_revenue REAL, monthly_costs REAL, staff_headcount INTEGER,
            avg_overtime_hours REAL, customer_satisfaction REAL,
            tier_cluster TEXT, attrition_risk_level TEXT,
            avg_daily_orders INTEGER DEFAULT 0,
            created_at TIMESTAMP DEFAULT CURRENT_TIMESTAMP)""")
        _safe_alter(conn, "ALTER TABLE outlets ADD COLUMN avg_daily_orders INTEGER DEFAULT 0")

        conn.execute("""CREATE TABLE IF NOT EXISTS staff (
            staff_id TEXT PRIMARY KEY, outlet_id TEXT, employee_name TEXT,
            role TEXT, monthly_salary REAL, weekly_overtime_hrs REAL,
            job_satisfaction INTEGER, employee_age INTEGER, tenure_years REAL,
            work_life_balance INTEGER, predicted_attrition_prob REAL,
            intervention_status TEXT DEFAULT 'Active')""")

        conn.execute("""CREATE TABLE IF NOT EXISTS inventory_records (
            record_id INTEGER PRIMARY KEY AUTOINCREMENT, outlet_id TEXT,
            sku_name TEXT, current_stock INTEGER, weekly_demand INTEGER,
            reorder_threshold INTEGER, stockout_risk_prob REAL,
            last_updated TIMESTAMP DEFAULT CURRENT_TIMESTAMP)""")

        conn.execute("""CREATE TABLE IF NOT EXISTS merged_datasets (
            id INTEGER PRIMARY KEY AUTOINCREMENT, agent_target TEXT, dataset_source TEXT,
            outlet_id TEXT, employee_age INTEGER, overtime_hours REAL,
            job_satisfaction INTEGER, attrition_target INTEGER, monthly_sales_usd REAL,
            operating_cost_usd REAL, tier_cluster_label INTEGER, sku_demand INTEGER,
            weather_impact_factor REAL, stockout_target INTEGER,
            created_at TIMESTAMP DEFAULT CURRENT_TIMESTAMP)""")

        # ── users table (Milestone 2: lockout + status columns) ──────────────
        conn.execute("""CREATE TABLE IF NOT EXISTS users (
            id INTEGER PRIMARY KEY AUTOINCREMENT, username TEXT UNIQUE,
            email TEXT UNIQUE, password_hash TEXT,
            security_question TEXT, security_answer_hash TEXT,
            role TEXT DEFAULT 'Franchise Owner',
            failed_attempts INTEGER DEFAULT 0,
            lock_until TIMESTAMP DEFAULT NULL,
            account_status TEXT DEFAULT 'active',
            created_at TIMESTAMP DEFAULT CURRENT_TIMESTAMP)""")
        # migrations for anyone upgrading from a Milestone 1 DB
        _safe_alter(conn, "ALTER TABLE users ADD COLUMN security_question TEXT")
        _safe_alter(conn, "ALTER TABLE users ADD COLUMN security_answer_hash TEXT")
        _safe_alter(conn, "ALTER TABLE users ADD COLUMN failed_attempts INTEGER DEFAULT 0")
        _safe_alter(conn, "ALTER TABLE users ADD COLUMN lock_until TIMESTAMP DEFAULT NULL")
        _safe_alter(conn, "ALTER TABLE users ADD COLUMN account_status TEXT DEFAULT 'active'")

        # ── OTP tracking (forgot-password Gmail OTP + resend cooldown) ───────
        conn.execute("""CREATE TABLE IF NOT EXISTS otp_codes (
            id INTEGER PRIMARY KEY AUTOINCREMENT,
            email TEXT,
            otp_hash TEXT,
            purpose TEXT DEFAULT 'reset',
            resend_count INTEGER DEFAULT 0,
            otp_next_allowed TIMESTAMP DEFAULT NULL,
            expires_at TIMESTAMP,
            consumed INTEGER DEFAULT 0,
            created_at TIMESTAMP DEFAULT CURRENT_TIMESTAMP)""")

        conn.execute("""CREATE TABLE IF NOT EXISTS ml_models (
            id INTEGER PRIMARY KEY AUTOINCREMENT,
            agent_name TEXT, model_name TEXT, r2_score REAL,
            rmse REAL, accuracy REAL, training_rows INTEGER,
            file_path TEXT, created_at TIMESTAMP DEFAULT CURRENT_TIMESTAMP)""")

        conn.execute("""CREATE TABLE IF NOT EXISTS notifications (
            id INTEGER PRIMARY KEY AUTOINCREMENT,
            channel TEXT, recipient TEXT, subject TEXT, message TEXT,
            status TEXT DEFAULT 'Sent',
            created_at TIMESTAMP DEFAULT CURRENT_TIMESTAMP)""")

        conn.execute("""CREATE TABLE IF NOT EXISTS chat_history (
            id INTEGER PRIMARY KEY AUTOINCREMENT,
            username TEXT NOT NULL, role TEXT NOT NULL, content TEXT NOT NULL,
            created_at TIMESTAMP DEFAULT CURRENT_TIMESTAMP)""")
        conn.commit()


# ──────────────────────────────────────────────────────────────────────────
# ML metrics
# ──────────────────────────────────────────────────────────────────────────
def save_ml_metrics(agent_name, model_name, r2, rmse, acc, rows, path):
    with get_conn() as conn:
        conn.execute("INSERT INTO ml_models "
                     "(agent_name,model_name,r2_score,rmse,accuracy,training_rows,file_path) "
                     "VALUES (?,?,?,?,?,?,?)",
                     (agent_name, model_name, r2, rmse, acc, rows, path))
        conn.commit()


# ──────────────────────────────────────────────────────────────────────────
# Chat history (Copilot)
# ──────────────────────────────────────────────────────────────────────────
def load_chat_history(username, conn_fn=None, limit=60):
    fn = conn_fn or get_conn
    with fn() as conn:
        rows = conn.execute(
            "SELECT role,content FROM chat_history WHERE username=? "
            "ORDER BY id DESC LIMIT ?", (username, limit)).fetchall()
    return [{"role": r[0], "content": r[1]} for r in reversed(rows)]


def save_chat_message(username, role, content, conn_fn=None):
    fn = conn_fn or get_conn
    with fn() as conn:
        conn.execute("INSERT INTO chat_history (username,role,content) VALUES (?,?,?)",
                     (username, role, content))
        conn.commit()


def clear_chat_history(username, conn_fn=None):
    fn = conn_fn or get_conn
    with fn() as conn:
        conn.execute("DELETE FROM chat_history WHERE username=?", (username,))
        conn.commit()


# ──────────────────────────────────────────────────────────────────────────
# Admin lifecycle: Add / Delete / Unlock (Section 9)
# ──────────────────────────────────────────────────────────────────────────
def admin_add_user(username, email, password_hash, role):
    with get_conn() as conn:
        conn.execute(
            "INSERT INTO users (username, email, password_hash, role, "
            "failed_attempts, lock_until, account_status) VALUES (?,?,?,?,0,NULL,'active')",
            (username, email, password_hash, role))
        conn.commit()


def admin_delete_user(user_id):
    with get_conn() as conn:
        conn.execute("DELETE FROM users WHERE id=?", (user_id,))
        conn.commit()


def admin_unlock_user(user_id):
    with get_conn() as conn:
        conn.execute(
            "UPDATE users SET failed_attempts=0, lock_until=NULL, "
            "account_status='active' WHERE id=?", (user_id,))
        conn.commit()


def list_users():
    with get_conn() as conn:
        return conn.execute(
            "SELECT id, username, email, role, failed_attempts, lock_until, "
            "account_status, created_at FROM users ORDER BY id DESC").fetchall()


# ──────────────────────────────────────────────────────────────────────────
# Progressive lockout (Section 5)
# ──────────────────────────────────────────────────────────────────────────
def get_user_lock_state(email_or_username):
    with get_conn() as conn:
        return conn.execute(
            "SELECT id, failed_attempts, lock_until, account_status FROM users "
            "WHERE email=? OR username=?", (email_or_username, email_or_username)).fetchone()


def register_failed_login(user_id, failed_attempts):
    """Apply the 3rd/4th/5th-attempt lockout ladder. Returns a user-facing message or None."""
    new_count = failed_attempts + 1
    now = datetime.datetime.utcnow()
    msg = None
    with get_conn() as conn:
        if new_count == 3:
            lock_until = now + datetime.timedelta(seconds=300)
            conn.execute("UPDATE users SET failed_attempts=?, lock_until=? WHERE id=?",
                         (new_count, lock_until.isoformat(), user_id))
            msg = "⏳ Account temporarily locked for 5 minutes due to 3 failed attempts."
        elif new_count == 4:
            lock_until = now + datetime.timedelta(seconds=900)
            conn.execute("UPDATE users SET failed_attempts=?, lock_until=? WHERE id=?",
                         (new_count, lock_until.isoformat(), user_id))
            msg = "⏳ Account temporarily locked for 15 minutes due to 4 failed attempts."
        elif new_count >= 5:
            conn.execute("UPDATE users SET failed_attempts=?, lock_until=NULL, "
                         "account_status='locked' WHERE id=?", (new_count, user_id))
            msg = ("❌ Account permanently locked due to 5 failed attempts. "
                   "Only the System Administrator can unlock this account via the Admin Dashboard.")
        else:
            conn.execute("UPDATE users SET failed_attempts=? WHERE id=?", (new_count, user_id))
        conn.commit()
    return msg


def reset_lockout_on_success(user_id):
    with get_conn() as conn:
        conn.execute("UPDATE users SET failed_attempts=0, lock_until=NULL WHERE id=?", (user_id,))
        conn.commit()


# ──────────────────────────────────────────────────────────────────────────
# OTP (Section 5.1 — resend cooldown 60s / 180s / 300s / 1hr)
# ──────────────────────────────────────────────────────────────────────────
def get_otp_state(email):
    with get_conn() as conn:
        return conn.execute(
            "SELECT id, resend_count, otp_next_allowed FROM otp_codes "
            "WHERE email=? AND purpose='reset' AND consumed=0 "
            "ORDER BY id DESC LIMIT 1", (email,)).fetchone()


def create_or_update_otp(email, otp_hash, resend_count, cooldown_seconds, ttl_seconds=600):
    now = datetime.datetime.utcnow()
    next_allowed = now + datetime.timedelta(seconds=cooldown_seconds)
    expires_at = now + datetime.timedelta(seconds=ttl_seconds)
    with get_conn() as conn:
        # invalidate previous unconsumed OTPs for this email
        conn.execute("UPDATE otp_codes SET consumed=1 WHERE email=? AND purpose='reset' AND consumed=0", (email,))
        conn.execute(
            "INSERT INTO otp_codes (email, otp_hash, purpose, resend_count, otp_next_allowed, expires_at) "
            "VALUES (?,?,?,?,?,?)",
            (email, otp_hash, "reset", resend_count, next_allowed.isoformat(), expires_at.isoformat()))
        conn.commit()


def consume_otp(otp_id):
    with get_conn() as conn:
        conn.execute("UPDATE otp_codes SET consumed=1 WHERE id=?", (otp_id,))
        conn.commit()


Overwriting db.py


In [16]:
%%writefile weather_context.py
"""
weather_context.py for FranchiseOps AI
Simulates local Indian city weather disruptions and logistics delays across franchise outlets.
"""
import random

CITY_WEATHER_REPORTS = {
    "Mumbai (MH)": {"status": "Heavy Monsoon Rain & Waterlogging", "temp_c": 28, "demand_impact_pct": -18.0, "supply_delay_days": 2, "attrition_stress": "High"},
    "Bengaluru (KA)": {"status": "Pleasant / Light Showers", "temp_c": 24, "demand_impact_pct": 12.0, "supply_delay_days": 0, "attrition_stress": "Normal"},
    "Delhi NCR (DL)": {"status": "Intense Summer Heatwave & Smog", "temp_c": 42, "demand_impact_pct": 15.0, "supply_delay_days": 1, "attrition_stress": "High"},
    "Hyderabad (TG)": {"status": "Clear & Warm", "temp_c": 33, "demand_impact_pct": 8.0, "supply_delay_days": 0, "attrition_stress": "Normal"},
    "Chennai (TN)": {"status": "Humid & Coastal Showers", "temp_c": 35, "demand_impact_pct": -5.0, "supply_delay_days": 1, "attrition_stress": "Medium"},
    "Pune (MH)": {"status": "Cloudy & Breezy", "temp_c": 26, "demand_impact_pct": 10.0, "supply_delay_days": 0, "attrition_stress": "Normal"},
    "Ahmedabad (GJ)": {"status": "Dry & High Heat", "temp_c": 40, "demand_impact_pct": -8.0, "supply_delay_days": 1, "attrition_stress": "Medium"},
    "Kolkata (WB)": {"status": "Thunderstorms & High Humidity", "temp_c": 32, "demand_impact_pct": -12.0, "supply_delay_days": 2, "attrition_stress": "High"}
}

def get_city_weather(city_name):
    for k, v in CITY_WEATHER_REPORTS.items():
        if k.lower() in city_name.lower() or city_name.lower() in k.lower():
            return {"city": k, **v}
    return {"city": city_name, "status": "Fair Weather Conditions", "temp_c": 30, "demand_impact_pct": 0.0, "supply_delay_days": 0, "attrition_stress": "Normal"}

def get_weather_report(port_name):
    return {"port": port_name, "status": "Normal Marine Conditions", "temp_c": 25, "wind_kt": 15, "delay_penalty_multiplier": 1.00}


Writing weather_context.py


In [18]:
%%writefile notifications.py
"""
FranchiseOps AI - notifications.py
Multi-channel alert center simulating SMS, Email, and In-App notifications stored in SQLite.
"""
from db import get_conn

def send_alert(channel, recipient, subject, message):
    with get_conn() as conn:
        conn.execute("INSERT INTO notifications (channel, recipient, subject, message, status) VALUES (?, ?, ?, ?, ?)",
                     (channel, recipient, subject, message, "Delivered"))
        conn.commit()
    print(f"[{channel.upper()}] To: {recipient} | Subject: {subject} | Status: Delivered")

def get_recent_alerts(limit=15):
    with get_conn() as conn:
        return conn.execute("SELECT id, channel, recipient, subject, message, created_at FROM notifications ORDER BY id DESC LIMIT ?", (limit,)).fetchall()


Overwriting notifications.py


In [20]:
%%writefile seed_data.py
"""
FranchiseOps AI - seed_data.py
Pre-seeds the database with 10 outlets (across the 6 core Indian retail cities:
Mumbai, Delhi NCR, Bengaluru, Hyderabad, Chennai, Pune), staff, and inventory data.
"""
from db import get_conn, init_db
from notifications import send_alert


def seed_all():
    init_db()
    with get_conn() as conn:
        # Seed Outlets — 10 outlets, avg_daily_orders added for Agent-2 KMeans tiering
        if not conn.execute("SELECT count(*) FROM outlets").fetchone()[0]:
            outlets = [
                ("OUT-101", "Mumbai Flagship Store",      "Mumbai (MH)",    145000, 112000, 24, 18.5, 4.2, "Tier 3 (At-Risk)",  "High Attrition",      210),
                ("OUT-102", "Bengaluru Tech Hub Cafe",    "Bengaluru (KA)", 285000, 165000, 32, 4.2,  4.8, "Tier 1 (Apex)",     "Low Attrition",       410),
                ("OUT-103", "Delhi NCR Metro Express",    "Delhi NCR (DL)", 210000, 155000, 28, 14.0, 4.5, "Tier 2 (Stable)",   "Moderate Attrition",  320),
                ("OUT-104", "Hyderabad Central Hub",      "Hyderabad (TG)", 125000, 118000, 18, 22.0, 3.8, "Tier 3 (At-Risk)",  "Critical Attrition",  180),
                ("OUT-105", "Chennai Coastal Kiosk",      "Chennai (TN)",   195000, 138000, 26, 6.5,  4.7, "Tier 1 (Apex)",     "Low Attrition",       360),
                ("OUT-106", "Pune IT Park Outlet",        "Pune (MH)",      172000, 129000, 22, 9.8,  4.4, "Tier 2 (Stable)",   "Low Attrition",       300),
                ("OUT-107", "Mumbai Suburban Express",    "Mumbai (MH)",    98000,  89000,  16, 26.0, 3.5, "Tier 3 (At-Risk)",  "Critical Attrition",  150),
                ("OUT-108", "Bengaluru Airport Kiosk",    "Bengaluru (KA)", 240000, 152000, 20, 8.0,  4.6, "Tier 1 (Apex)",     "Low Attrition",       380),
                ("OUT-109", "Delhi NCR Connaught Outlet", "Delhi NCR (DL)", 158000, 121000, 21, 16.5, 4.1, "Tier 2 (Stable)",   "Moderate Attrition",  260),
                ("OUT-110", "Chennai Mall Express",       "Chennai (TN)",   112000, 96000,  17, 20.0, 3.9, "Tier 3 (At-Risk)",  "High Attrition",      190),
            ]
            conn.executemany(
                "INSERT INTO outlets (outlet_id, outlet_name, city, monthly_revenue, monthly_costs, "
                "staff_headcount, avg_overtime_hours, customer_satisfaction, tier_cluster, "
                "attrition_risk_level, avg_daily_orders) VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?)",
                outlets)

        # Seed Staff
        if not conn.execute("SELECT count(*) FROM staff").fetchone()[0]:
            staff = [
                ("ST-5001", "OUT-101", "Marcus Vance",   "Shift Supervisor", 3920.0, 21.0, 2, 32, 4.5, 2, 0.82, "Retention Bonus Offered"),
                ("ST-5002", "OUT-101", "Elena Rostova",  "Barista / Cashier", 2880.0, 19.5, 2, 26, 2.0, 2, 0.79, "Schedule Adjusted"),
                ("ST-5003", "OUT-102", "David Chen",     "Store Manager",    5120.0, 3.5,  5, 41, 8.5, 4, 0.12, "Stable"),
                ("ST-5004", "OUT-104", "Samantha Diaz",  "Kitchen Lead",     3360.0, 24.5, 1, 29, 3.0, 1, 0.89, "Immediate Review Required"),
                ("ST-5005", "OUT-105", "James Wilson",   "Team Lead",        4000.0, 5.0,  4, 36, 6.0, 3, 0.18, "Stable"),
                ("ST-5006", "OUT-107", "Priya Nair",     "Barista / Cashier", 2650.0, 27.0, 1, 24, 1.0, 1, 0.91, "Immediate Review Required"),
                ("ST-5007", "OUT-108", "Arjun Rao",      "Shift Supervisor", 3800.0, 6.0,  4, 30, 5.5, 3, 0.15, "Stable"),
            ]
            conn.executemany(
                "INSERT INTO staff (staff_id, outlet_id, employee_name, role, monthly_salary, "
                "weekly_overtime_hrs, job_satisfaction, employee_age, tenure_years, work_life_balance, "
                "predicted_attrition_prob, intervention_status) VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?)",
                staff)

        # Seed Inventory
        if not conn.execute("SELECT count(*) FROM inventory_records").fetchone()[0]:
            inventory = [
                ("OUT-101", "Premium Coffee Beans (Kg)", 140, 320, 180, 0.84),
                ("OUT-101", "Organic Milk Syrups (L)",    85, 190, 100, 0.78),
                ("OUT-102", "Premium Coffee Beans (Kg)", 580, 450, 250, 0.12),
                ("OUT-104", "Eco-Packaging Cups (Box)",   40, 210, 150, 0.91),
                ("OUT-105", "Artisan Tea Blends (Kg)",   310, 220, 140, 0.15),
                ("OUT-107", "Premium Coffee Beans (Kg)",  55, 200, 130, 0.88),
                ("OUT-108", "Organic Milk Syrups (L)",   270, 210, 120, 0.20),
            ]
            conn.executemany(
                "INSERT INTO inventory_records (outlet_id, sku_name, current_stock, weekly_demand, "
                "reorder_threshold, stockout_risk_prob) VALUES (?, ?, ?, ?, ?, ?)",
                inventory)
            conn.commit()

    send_alert("Email", "franchisee@franchiseops.ai", "Franchise Operations Initialized",
               "Database seeded with 10 outlets across 6 Indian retail hubs, staff logs, and inventory benchmarks.")
    print("✅ Database pre-seeded successfully — 10 outlets ready for KMeans tiering.")


Overwriting seed_data.py


In [22]:
%%writefile admin_dash.py
"""admin_dash.py — Admin Dashboard for FranchiseOps AI (Milestone 2)
Adds: Add User / Delete User / Unlock Account lifecycle controls (Section 9),
and a dedicated ML Model Card tab showing all 3 agents' metrics + KMeans tiering.
"""
import subprocess, datetime
import streamlit as st
import pandas as pd
import plotly.express as px
from db import get_conn, admin_add_user, admin_delete_user, admin_unlock_user, list_users
from auth import hash_txt, password_strength
from notifications import get_recent_alerts
from ui_theme import render_card, COLORS

_APP_START = datetime.datetime.now()


def _smi(query):
    try:
        r = subprocess.run(
            ["nvidia-smi", f"--query-gpu={query}", "--format=csv,noheader,nounits"],
            capture_output=True, text=True, timeout=3)
        return r.stdout.strip()
    except Exception:
        return "N/A"


def render_admin_dashboard(project="franchiseops"):
    render_card('<h3 style="margin:0;">🛡️ Admin Dashboard — System Intelligence</h3>')

    tab_health, tab_users, tab_models, tab_alerts = st.tabs(
        ["⚙️ System Health", "👥 User Management", "📈 ML Model Card", "🔔 Alert Log"])

    # ── System Health ─────────────────────────────────────────────────────
    with tab_health:
        gpu_mem = _smi("memory.used")
        gpu_tot = _smi("memory.total")
        gpu_util = _smi("utilization.gpu")
        uptime = str(datetime.datetime.now() - _APP_START).split(".")[0]
        h1, h2, h3, h4 = st.columns(4)
        for col, icon, label, val in [
            (h1, "🖥️", "GPU VRAM Used", f"{gpu_mem} / {gpu_tot} MB"),
            (h2, "⚡", "GPU Utilization", f"{gpu_util}%"),
            (h3, "🕒", "App Uptime", uptime),
            (h4, "✅", "LLM Status", "Active" if gpu_mem != "N/A" else "Standby"),
        ]:
            col.markdown(
                f'<div class="pn-card" style="text-align:center;padding:14px;">'
                f'<div style="font-size:26px;">{icon}</div>'
                f'<h3 style="margin:6px 0 2px;font-size:1.1rem;">{val}</h3>'
                f'<p style="margin:0;color:{COLORS["text_muted"]};font-size:12px;">{label}</p>'
                f'</div>', unsafe_allow_html=True)

    # ── User Management: Add / Delete / Unlock ───────────────────────────
    with tab_users:
        st.markdown(f'<h4 style="color:{COLORS["text_heading"]};margin:0 0 8px;">➕ Add User</h4>',
                    unsafe_allow_html=True)
        with st.form("add_user_form", clear_on_submit=True):
            ac1, ac2 = st.columns(2)
            new_username = ac1.text_input("Username")
            new_email = ac2.text_input("Email")
            ac3, ac4 = st.columns(2)
            new_password = ac3.text_input("Initial Password", type="password")
            new_role = ac4.selectbox("Role", ["Admin", "Franchise Owner",
                                               "Regional Operations Manager",
                                               "Store Manager", "Supply Chain Analyst"])
            submitted = st.form_submit_button("Create User")
            if submitted:
                _badge, allowed, msg = password_strength(new_password or "")
                if not (new_username and new_email and new_password):
                    st.warning("Please fill out all fields.")
                elif not allowed:
                    st.warning(msg)
                else:
                    try:
                        admin_add_user(new_username, new_email, hash_txt(new_password), new_role)
                        st.success(f"✅ User '{new_username}' created with role [{new_role}].")
                        st.rerun()
                    except Exception:
                        st.error("Could not create user — username or email may already exist.")

        st.markdown("---")
        st.markdown(f'<h4 style="color:{COLORS["text_heading"]};margin:0 0 8px;">👥 Existing Users</h4>',
                    unsafe_allow_html=True)

        rows = list_users()
        cols = ["id", "username", "email", "role", "failed_attempts", "lock_until", "account_status", "created_at"]
        users_df = pd.DataFrame(rows, columns=cols) if rows else pd.DataFrame(columns=cols)

        if users_df.empty:
            st.info("No users registered yet.")
        else:
            for _, row in users_df.iterrows():
                uc1, uc2, uc3, uc4, uc5 = st.columns([2, 2, 1.4, 1, 1])
                uc1.markdown(f"**{row['username']}**  \n<span style='font-size:12px;color:{COLORS['text_muted']}'>{row['email']}</span>",
                             unsafe_allow_html=True)
                uc2.markdown(f'<span style="color:#0066cc;font-weight:600;">[{row["role"]}]</span>',
                             unsafe_allow_html=True)
                is_locked = row["account_status"] == "locked" or (row["failed_attempts"] or 0) >= 3
                status_badge = "🔒 Locked" if is_locked else "🟢 Active"
                uc3.markdown(status_badge)
                with uc4:
                    if is_locked:
                        if st.button("🔓 Unlock", key=f"unlock_{row['id']}"):
                            admin_unlock_user(row["id"])
                            st.success(f"✅ User account unlocked successfully.")
                            st.rerun()
                with uc5:
                    if st.button("🗑️", key=f"del_user_{row['id']}", help=f"Delete {row['username']}"):
                        admin_delete_user(row["id"])
                        st.success(f"Deleted {row['username']}")
                        st.rerun()

    # ── ML Model Card ─────────────────────────────────────────────────────
    with tab_models:
        st.markdown(f'<h4 style="color:{COLORS["text_heading"]};margin:0 0 8px;">📈 Champion Model Metrics — All 3 Agents</h4>',
                    unsafe_allow_html=True)
        with get_conn() as conn:
            try:
                ml_df = pd.read_sql(
                    "SELECT agent_name, model_name, r2_score, accuracy, rmse, "
                    "training_rows, created_at FROM ml_models ORDER BY id DESC", conn)
            except Exception:
                ml_df = pd.DataFrame()
        if ml_df.empty:
            st.info("No model training records found. Run train_m2_franchise.py first.")
        else:
            st.dataframe(ml_df, use_container_width=True, hide_index=True)
            latest_per_agent = ml_df.sort_values("created_at").groupby("agent_name").tail(1)
            mcols = st.columns(min(4, len(latest_per_agent))) if len(latest_per_agent) else []
            for col, (_, r) in zip(mcols, latest_per_agent.iterrows()):
                metric_val = r["accuracy"] if r["accuracy"] not in (None, 0) else r["r2_score"]
                col.metric(r["agent_name"], f"{metric_val:.3f}" if pd.notna(metric_val) else "—", r["model_name"])

    # ── Alert Log ──────────────────────────────────────────────────────────
    with tab_alerts:
        filt = st.selectbox("Filter by type", ["All", "In-App", "Email", "SMS"], key="admin_alert_filt")
        alerts = get_recent_alerts(50)
        for a in alerts:
            if filt != "All" and a[1].lower() != filt.lower():
                continue
            badge = {"email": "#ffd803", "sms": "#f87171", "in-app": "#34d399"}.get(a[1].lower(), "#bae8e8")
            st.markdown(
                f'<div style="border-left:4px solid {badge};padding:4px 10px;margin:3px 0;'
                f'font-size:13px;"><b>[{a[1].upper()}]</b> {a[3]} '
                f'<span style="color:{COLORS["text_muted"]};float:right;">{a[4]}</span></div>',
                unsafe_allow_html=True)


Overwriting admin_dash.py


In [24]:
%%writefile agent2_franchise.py
"""
agent2_franchise.py — Enriched Agent 2: Outlet Territory Clustering & City Weather
New features: City demand surge chart, revenue vs weather scatter, AI territory advisory.
Extended Indian cities + global franchise locations.
"""
import numpy as np
import pandas as pd
import streamlit as st
import plotly.express as px
from ui_theme import render_card, COLORS
from db import get_conn
from weather_context import get_city_weather
from llm_engine_franchise import orchestrate_3_agents_query

# ── Full outlet / city list (heavy India coverage) ───────────────────────────
INDIA_CITIES = [
    "Mumbai (MH)", "Delhi (DL)", "Bengaluru (KA)", "Hyderabad (TS)",
    "Chennai (TN)", "Pune (MH)", "Kolkata (WB)", "Ahmedabad (GJ)",
    "Jaipur (RJ)", "Surat (GJ)", "Lucknow (UP)", "Chandigarh (PB)",
    "Bhopal (MP)", "Indore (MP)", "Nagpur (MH)", "Coimbatore (TN)",
    "Kochi (KL)", "Visakhapatnam (AP)", "Patna (BR)", "Ranchi (JH)",
]
GLOBAL_CITIES = [
    "Chicago (IL)", "Los Angeles (CA)", "New York (NY)", "Houston (TX)",
    "London (UK)", "Dubai (AE)", "Singapore (SG)",
]
ALL_CITIES = INDIA_CITIES + GLOBAL_CITIES

# 4-tier scheme (Section 7): Excellent / Good / Needs Attention / Critical
TIER_ORDER  = ["Excellent", "Good", "Needs Attention", "Critical"]
TIER_COLORS = {"Excellent": "#34d399", "Good": "#22d3ee",
               "Needs Attention": "#fbbf24", "Critical": "#f87171"}


def _tier_label_map(km):
    """Ranks the fitted KMeans clusters by (revenue + orders) descending and maps
    each cluster index to Excellent/Good/Needs Attention/Critical — consistent with
    how train_m2_franchise.py assigned tiers to the 10 seeded outlets."""
    centers = km.cluster_centers_  # [:,0]=avg_daily_revenue, [:,1]=avg_daily_orders
    rev, orders = centers[:, 0], centers[:, 1]
    rev_n = rev / (rev.max() if rev.max() else 1)
    ord_n = orders / (orders.max() if orders.max() else 1)
    score = rev_n + ord_n
    ranked_clusters = np.argsort(-score)
    return {cl: TIER_ORDER[rank] if rank < len(TIER_ORDER) else TIER_ORDER[-1]
            for rank, cl in enumerate(ranked_clusters)}


def render_agent2_franchise(agent2_c, agent2_r, username, db_stats, a1_ctx, a3_ctx,
                             send_alert, confidence_band):
    render_card('<h3 style="margin:0;">🏬 Agent 2: Outlet Territory Clustering</h3>')

    with get_conn() as conn:
        try:
            out_df = pd.read_sql("SELECT * FROM outlets", conn)
        except Exception:
            out_df = pd.DataFrame()

    c1, c2 = st.columns([1.3, 1])
    with c1:
        if not out_df.empty:
            st.dataframe(
                out_df[["outlet_id", "outlet_name", "city",
                        "monthly_revenue", "avg_daily_orders", "tier_cluster"]],
                use_container_width=True, hide_index=True)
            fig = px.scatter(
                out_df, x="avg_daily_orders", y="monthly_revenue",
                color="tier_cluster", size="staff_headcount",
                hover_name="outlet_name",
                title="Revenue vs Order Count Clustering",
                category_orders={"tier_cluster": TIER_ORDER},
                color_discrete_map=TIER_COLORS)
            fig.update_layout(paper_bgcolor="rgba(0,0,0,0)", plot_bgcolor="rgba(0,0,0,0)",
                              height=280, margin=dict(l=10, r=10, t=40, b=10))
            st.plotly_chart(fig, use_container_width=True)

    with c2:
        render_card('<h4 style="margin:0 0 10px;">Simulate New Outlet</h4>')
        city_sel   = st.selectbox("City", ALL_CITIES)
        new_rev    = st.number_input("Monthly Revenue (₹)", 80000.0, 2000000.0, 380000.0, step=10000.0)
        new_cost   = st.number_input("Monthly Costs (₹)", 50000.0, 1500000.0, 260000.0, step=10000.0)
        new_hc     = st.slider("Staff Headcount", 5, 80, 22)
        new_orders = st.slider("Avg Daily Orders", 50, 500, 250)
        if st.button("⚡ Predict Tier Cluster", key="btn_predict_tier"):
            avg_daily_rev = new_rev / 30.0
            if agent2_c:
                idx = agent2_c.predict([[avg_daily_rev, new_orders]])[0]
                tier = _tier_label_map(agent2_c).get(idx, "Good")
            else:
                margin = new_rev - new_cost
                tier = ("Excellent" if new_rev > 500000 else
                        "Needs Attention" if margin < 40000 else "Good")
            st.markdown(
                f'<div style="background:{TIER_COLORS[tier]};padding:14px;border-radius:12px;'
                f'border:2px solid #272343;font-weight:700;font-size:16px;color:#0b0e14;">'
                f'{tier}</div>', unsafe_allow_html=True)

    st.markdown("---")
    tab_demand, tab_corr, tab_ai = st.tabs(
        ["📊 City Demand Surge", "📈 Revenue vs Weather", "🤖 AI Advisory"])

    # ── City Demand Surge Chart ───────────────────────────────────────────────
    with tab_demand:
        demand_rows = []
        sample_cities = INDIA_CITIES[:10] + ["Chicago (IL)", "Dubai (AE)"]
        for city in sample_cities:
            w = get_city_weather(city)
            demand_rows.append({
                "City": city.split(" (")[0],
                "Demand Impact %": w.get("demand_impact_pct", 0),
                "Weather": w.get("status", "Normal"),
            })
        d_df = pd.DataFrame(demand_rows).sort_values("Demand Impact %", ascending=False)
        fig2 = px.bar(d_df, x="City", y="Demand Impact %", color="Demand Impact %",
                      color_continuous_scale=["#34d399", "#ffd803", "#f87171"],
                      title="City Demand Surge / Weather Impact")
        fig2.update_layout(paper_bgcolor="rgba(0,0,0,0)", plot_bgcolor="rgba(0,0,0,0)",
                           height=320, margin=dict(l=10, r=10, t=40, b=80),
                           xaxis_tickangle=-35)
        st.plotly_chart(fig2, use_container_width=True)

    # ── Revenue vs Weather Correlation ────────────────────────────────────────
    with tab_corr:
        if not out_df.empty and "city" in out_df.columns:
            out_df["demand_impact"] = out_df["city"].apply(
                lambda c: get_city_weather(c).get("demand_impact_pct", 0))
            fig3 = px.scatter(out_df, x="demand_impact", y="monthly_revenue",
                              color="tier_cluster", size="staff_headcount",
                              hover_name="outlet_name",
                              trendline="ols",
                              title="Revenue vs Weather Demand Impact",
                              category_orders={"tier_cluster": TIER_ORDER},
                              color_discrete_map=TIER_COLORS)
            fig3.update_layout(paper_bgcolor="rgba(0,0,0,0)", plot_bgcolor="rgba(0,0,0,0)",
                               height=300, margin=dict(l=10, r=10, t=40, b=10))
            st.plotly_chart(fig3, use_container_width=True)
        else:
            st.info("Outlet data with city weather not available.")

    # ── AI Territory Advisory ─────────────────────────────────────────────────
    with tab_ai:
        if st.button("🤖 Get AI Territory Advisory", key="btn_a2f_advisory"):
            a2_ctx = {"city": city_sel, "revenue": new_rev, "costs": new_cost, "headcount": new_hc,
                      "orders": new_orders, "weather": get_city_weather(city_sel)}
            with st.spinner("Generating advisory (~2 sec)..."):
                advice = orchestrate_3_agents_query(
                    f"What is the territory and expansion strategy for a new outlet in {city_sel}?",
                    a1_ctx, a2_ctx, a3_ctx, db_stats)
            st.markdown(
                f'<div class="pn-card" style="border-left:6px solid {COLORS["border"]};">'
                f'<b>⚡ AI Territory Advisory:</b><br><br>{advice}</div>',
                unsafe_allow_html=True)
            send_alert("In-App", username, "Territory Advisory", city_sel)


Overwriting agent2_franchise.py


In [26]:
%%writefile agent3_franchise.py
"""
agent3_franchise.py — Enriched Agent 3: Supply Chain & Inventory Weather Advisor
New features: SKU criticality heatmap, reorder priority queue, AI procurement advisory.
"""
import numpy as np
import pandas as pd
import streamlit as st
import plotly.express as px
from ui_theme import render_card, COLORS
from db import get_conn
from weather_context import get_city_weather
from llm_engine_franchise import orchestrate_3_agents_query, generate_json
from notifications import send_alert

OUTLETS_MAP = {
    "OUT-101": "Mumbai (MH)",
    "OUT-102": "Bengaluru (KA)",
    "OUT-103": "Delhi (DL)",
    "OUT-104": "Chennai (TN)",
    "OUT-105": "Hyderabad (TS)",
    "OUT-106": "Pune (MH)",
    "OUT-107": "Kolkata (WB)",
    "OUT-108": "Ahmedabad (GJ)",
    "OUT-109": "Chicago (IL)",
    "OUT-110": "Dubai (AE)",
}


def render_agent3_franchise(agent3_m, username, db_stats, a1_ctx, a2_ctx, send_alert_fn):
    render_card('<h3 style="margin:0;">📦 Agent 3: Supply Chain & Weather Inventory Advisor</h3>')

    c1, c2 = st.columns(2)
    with c1:
        sel_out = st.selectbox("Outlet", list(OUTLETS_MAP.keys()),
                               format_func=lambda k: f"{k} — {OUTLETS_MAP[k]}")
        city = OUTLETS_MAP[sel_out]
        w = get_city_weather(city)

    with c2:
        render_card(
            f"<b>📍 City:</b> {city}<br>"
            f"<b>Weather:</b> {w['status']} ({w.get('temp_f', 'N/A')}°F)<br>"
            f"<b>Demand Impact:</b> <b>{w['demand_impact_pct']:+.1f}%</b><br>"
            f"<b>Supply Delay:</b> +{w.get('supply_delay_days', 1)} days", alt=True)

    st.markdown("---")
    tab_heat, tab_queue, tab_ai = st.tabs(
        ["🌡️ SKU Heatmap", "📋 Reorder Queue", "🤖 AI Procurement"])

    # ── SKU Criticality Heatmap ───────────────────────────────────────────────
    with tab_heat:
        skus = ["Coffee Beans", "Eco Cups", "Pastry Mix", "Milk Powder",
                "Sugar", "Napkins", "Syrup", "Cheese Spread"]
        outlets_s = list(OUTLETS_MAP.keys())[:6]
        np.random.seed(42)
        base = np.random.uniform(0.1, 0.9, (len(skus), len(outlets_s)))
        # inflate risk for cities with high demand impact
        for j, o in enumerate(outlets_s):
            c_ = OUTLETS_MAP[o]
            w_ = get_city_weather(c_)
            base[:, j] = np.clip(base[:, j] + w_["demand_impact_pct"] / 200, 0, 1)

        heat_df = pd.DataFrame(np.round(base, 2), index=skus, columns=outlets_s)
        fig = px.imshow(heat_df, text_auto=True, aspect="auto",
                        color_continuous_scale=["#34d399", "#ffd803", "#f87171"],
                        title="SKU Stockout Risk (0=Safe, 1=Critical)")
        fig.update_layout(paper_bgcolor="rgba(0,0,0,0)", height=340,
                          margin=dict(l=10, r=10, t=40, b=10))
        st.plotly_chart(fig, use_container_width=True)

    # ── Reorder Priority Queue ────────────────────────────────────────────────
    with tab_queue:
        rows = []
        for o, c_ in list(OUTLETS_MAP.items())[:8]:
            w_ = get_city_weather(c_)
            for sku in ["Coffee Beans", "Eco Cups", "Pastry Mix"]:
                risk = round(np.clip(0.3 + w_["demand_impact_pct"] / 150 + np.random.uniform(0, 0.3), 0, 1), 2)
                rows.append({
                    "Outlet": o, "City": c_.split(" (")[0], "SKU": sku,
                    "Stockout Risk": risk,
                    "Urgency": "🔴 Immediate" if risk > 0.7 else ("🟡 Soon" if risk > 0.45 else "🟢 OK"),
                    "Reorder Qty": int(risk * 500 + 100),
                })
        q_df = pd.DataFrame(rows).sort_values("Stockout Risk", ascending=False).head(10).reset_index(drop=True)
        q_df.index += 1
        st.dataframe(q_df, use_container_width=True)

    # ── AI Procurement Advisory ───────────────────────────────────────────────
    with tab_ai:
        if st.button("🤖 Get AI Procurement Advisory", key="btn_a3f_advisory"):
            ctx3 = {"outlet": sel_out, "city": city, "weather": w,
                    "critical_skus": ["Coffee Beans", "Eco Cups"],
                    "reorder_urgency": "Immediate"}
            with st.spinner("Generating advisory (~2 sec)..."):
                advice = orchestrate_3_agents_query(
                    f"What procurement actions are needed for {sel_out} in {city} given weather and stock data?",
                    a1_ctx, a2_ctx, ctx3, db_stats)
            st.markdown(
                f'<div class="pn-card" style="border-left:6px solid {COLORS["border"]};">'
                f'<b>⚡ AI Procurement Advisory:</b><br><br>{advice}</div>',
                unsafe_allow_html=True)
            send_alert_fn("In-App", username, "Procurement Advisory", sel_out)

        if st.button("📋 Generate JSON Reorder Plan", key="btn_reorder_json"):
            with st.spinner("Generating reorder plan (~2 sec)..."):
                plan = generate_json(
                    f"Outlet {sel_out} in {city}. Weather demand surge: {w['demand_impact_pct']:+.1f}%. "
                    f"Supply delay: {w.get('supply_delay_days', 1)} days. Critical SKUs: Coffee Beans, Eco Cups.",
                    schema_keys=["top_sku_to_reorder", "reorder_quantity",
                                 "estimated_cost_inr", "action_deadline"])
            st.json(plan)


Overwriting agent3_franchise.py


## Step 5 — Initialise Database & Seed Sample Data


In [27]:
import db, seed_data
db.init_db()
seed_data.seed_all()


[EMAIL] To: franchisee@franchiseops.ai | Subject: Franchise Operations Initialized | Status: Delivered
✅ Database pre-seeded successfully — 10 outlets ready for KMeans tiering.


## Step 6 — Train ML Agents


In [29]:
%%writefile train_m2_franchise.py
"""
train_m2_franchise.py — FranchiseOps AI (Milestone 2)
Multi-Algorithm Comparison (5+ per agent, per Section 7):
  Agent 1 (Attrition, classification): LogisticRegression, RandomForest, GradientBoosting,
           SVC(RBF), DecisionTreeClassifier, AdaBoostClassifier, KNeighborsClassifier → best ROC-AUC
  Agent 2 (Revenue, regression): RandomForest, GradientBoosting, ExtraTrees, Ridge,
           DecisionTreeRegressor, AdaBoostRegressor, KNeighborsRegressor → best R²
           + KMeans(k=4) tiering of the 10 seeded outlets → Excellent / Good / Needs Attention / Critical
  Agent 3 (Inventory demand, regression): same regressor family → best R²

Datasets pulled via kagglehub-style Kaggle API download using the exact slugs from Section 7.1;
falls back to a clearly-labeled synthetic dataset if Kaggle credentials aren't configured.
"""
import os, joblib, numpy as np, pandas as pd
from sklearn.ensemble import (RandomForestClassifier, GradientBoostingClassifier,
                               ExtraTreesClassifier, RandomForestRegressor,
                               GradientBoostingRegressor, ExtraTreesRegressor,
                               AdaBoostClassifier, AdaBoostRegressor)
from sklearn.tree import DecisionTreeClassifier, DecisionTreeRegressor
from sklearn.neighbors import KNeighborsClassifier, KNeighborsRegressor
from sklearn.linear_model import LogisticRegression, Ridge
from sklearn.svm import SVC
from sklearn.cluster import KMeans
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score, accuracy_score, r2_score, mean_squared_error, silhouette_score
from sklearn.calibration import CalibratedClassifierCV
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from config import (KAGGLE_USERNAME, KAGGLE_KEY, KAGGLE_CACHE_DIR, MODELS_DIR,
                    AGENT1_MODEL_PATH, AGENT2_MODEL_PATH, AGENT2_REG_PATH,
                    AGENT3_MODEL_PATH, KMEANS_MODEL_PATH)
from db import get_conn, save_ml_metrics, init_db

# Tier labels — EXACTLY matching Infosys spec, best (highest revenue/orders) → worst
TIER_ORDER = ["Excellent", "Good", "Needs Attention", "Critical"]


def kaggle_download(slug, filename, dest=KAGGLE_CACHE_DIR):
    """Pulls via kagglehub.dataset_download (per Section 7.1), falling back to the
    classic Kaggle API, then to None (→ synthetic data) if neither works."""
    target = os.path.join(dest, filename)

    def _clean_df(df):
        if df is not None:
            df.columns = df.columns.astype(str).str.strip().str.lstrip('\ufeff')
        return df

    if os.path.exists(target):
        print(f"  📂 Cache hit: {filename}")
        try:
            return _clean_df(pd.read_csv(target, encoding="latin-1", on_bad_lines="skip"))
        except Exception:
            pass
    if not (KAGGLE_USERNAME and KAGGLE_KEY):
        print(f"  ℹ️  No Kaggle creds — synthetic fallback")
        return None

    os.environ.update({"KAGGLE_USERNAME": KAGGLE_USERNAME, "KAGGLE_KEY": KAGGLE_KEY})

    # ── Primary: kagglehub.dataset_download (Section 7.1) ──────────────────
    try:
        import kagglehub
        print(f"  ⬇️  kagglehub.dataset_download({slug}) …")
        dl_path = kagglehub.dataset_download(slug)
        candidate = os.path.join(dl_path, filename)
        if not os.path.exists(candidate):
            csvs = [f for f in os.listdir(dl_path) if f.endswith(".csv")]
            candidate = os.path.join(dl_path, csvs[0]) if csvs else None
        if candidate and os.path.exists(candidate):
            import shutil
            shutil.copy(candidate, target)
            df = _clean_df(pd.read_csv(target, encoding="latin-1", on_bad_lines="skip"))
            print(f"  ✅ Loaded {len(df)} rows via kagglehub")
            return df
    except Exception as e:
        print(f"  ⚠️  kagglehub failed ({e}) — trying classic Kaggle API …")

    # ── Secondary: classic Kaggle API (in case kagglehub isn't installed/available) ──
    try:
        from kaggle.api.kaggle_api_extended import KaggleApi
        api = KaggleApi()
        api.authenticate()
        api.dataset_download_files(slug, path=dest, unzip=True, quiet=False)
        if os.path.exists(target):
            df = _clean_df(pd.read_csv(target, encoding="latin-1", on_bad_lines="skip"))
            print(f"  ✅ Loaded {len(df)} rows via classic Kaggle API")
            return df
        csvs = [f for f in os.listdir(dest) if f.endswith(".csv")]
        if csvs:
            df = _clean_df(pd.read_csv(os.path.join(dest, csvs[0]), encoding="latin-1", on_bad_lines="skip"))
            print(f"  ✅ Loaded {csvs[0]}: {len(df)} rows via classic Kaggle API")
            return df
    except Exception as e:
        print(f"  ⚠️  Classic Kaggle API also failed ({e}) — synthetic fallback")
    return None


def compare_classifiers(models_dict, X_tr, X_te, y_tr, y_te, agent_name, save_path):
    print(f"\n  🔬 {agent_name} — Algorithm Comparison ({len(models_dict)} models):")
    best_name, best_model, best_auc = None, None, -np.inf
    for name, base in models_dict.items():
        needs_calibration = not hasattr(base, "predict_proba") or name in ("SVC_RBF",)
        model = CalibratedClassifierCV(base, cv=2, method="sigmoid") if needs_calibration else base
        model.fit(X_tr, y_tr)
        proba = model.predict_proba(X_te)[:, 1]
        auc = float(roc_auc_score(y_te, proba))
        acc = float(accuracy_score(y_te, model.predict(X_te)))
        print(f"    {name:28s} ROC-AUC={auc:.4f}  Acc={acc*100:.1f}%")
        save_ml_metrics(agent_name, name, auc, 0.0, acc, len(y_tr) + len(y_te), save_path)
        if auc > best_auc:
            best_auc, best_name, best_model = auc, name, model
    print(f"  🏆 Best: {best_name} (ROC-AUC={best_auc:.4f})")
    joblib.dump(best_model, save_path)
    return best_model, best_name, best_auc


def compare_regressors(models_dict, X_tr, X_te, y_tr, y_te, agent_name, save_path):
    print(f"\n  🔬 {agent_name} — Algorithm Comparison ({len(models_dict)} models):")
    best_name, best_model, best_r2 = None, None, -np.inf
    for name, model in models_dict.items():
        model.fit(X_tr, y_tr)
        p = model.predict(X_te)
        r2 = float(r2_score(y_te, p))
        rmse = float(np.sqrt(mean_squared_error(y_te, p)))
        print(f"    {name:28s} R²={r2:.4f}  RMSE={rmse:.2f}")
        save_ml_metrics(agent_name, name, r2, rmse, 0.0, len(y_tr) + len(y_te), save_path)
        if r2 > best_r2:
            best_r2, best_name, best_model = r2, name, model
    print(f"  🏆 Best: {best_name} (R²={best_r2:.4f})")
    joblib.dump(best_model, save_path)
    return best_model, best_name, best_r2


def generate_datasets(n=2000, seed=42):
    init_db()
    rng = np.random.default_rng(seed)

    # ── Agent 1: Workforce Attrition (2 datasets per Section 7.1) ──────────
    raw1 = kaggle_download("pavansubhasht/ibm-hr-analytics-attrition-dataset",
                           "WA_Fn-UseC_-HR-Employee-Attrition.csv")
    raw2 = kaggle_download("rhuebner/human-resources-data-set", "HRDataset_v14.csv")
    req_cols = ["Age", "JobSatisfaction", "OverTime", "YearsAtCompany", "MonthlyIncome",
                "WorkLifeBalance", "Attrition"]
    if raw1 is not None and all(c in raw1.columns for c in req_cols):
        raw1 = raw1[req_cols].dropna().head(n)
        a1 = pd.DataFrame({
            "age":          raw1["Age"].astype(int).values,
            "satisfaction": raw1["JobSatisfaction"].astype(int).values,
            "overtime":     (raw1["OverTime"] == "Yes").astype(int).values,
            "tenure_yrs":   raw1["YearsAtCompany"].astype(int).values,
            "income":       raw1["MonthlyIncome"].astype(float).values,
            "worklife":     raw1["WorkLifeBalance"].astype(int).values,
            "attrition":    (raw1["Attrition"] == "Yes").astype(int).values,
        })
    else:
        n1 = n
        a1 = pd.DataFrame({
            "age":          rng.integers(18, 62, n1),
            "satisfaction": rng.integers(1, 5, n1),
            "overtime":     rng.choice([0, 1], n1, p=[0.72, 0.28]),
            "tenure_yrs":   rng.integers(0, 20, n1),
            "income":       rng.uniform(20000, 100000, n1),
            "worklife":     rng.integers(1, 4, n1),
        })
        p_attr = (a1["overtime"] * 0.35 + (5 - a1["satisfaction"]) / 4 * 0.35 +
                  (1 - a1["tenure_yrs"] / 20) * 0.30)
        a1["attrition"] = (p_attr > 0.55).astype(int)

    # Augment with HRDataset_v14 (2nd Agent-1 dataset per Section 7.1) if columns are usable
    if raw2 is not None:
        try:
            r2 = raw2.copy()
            r2_map = pd.DataFrame({
                "age":          rng.integers(22, 60, len(r2)),  # DOB parsing varies too much across exports
                "satisfaction": pd.to_numeric(r2.get("EmpSatisfaction"), errors="coerce").fillna(3).astype(int).clip(1, 5),
                "overtime":     (pd.to_numeric(r2.get("Absences"), errors="coerce").fillna(0) >
                                 pd.to_numeric(r2.get("Absences"), errors="coerce").fillna(0).median()).astype(int),
                "tenure_yrs":   rng.integers(0, 15, len(r2)),
                "income":       pd.to_numeric(r2.get("Salary"), errors="coerce").fillna(45000).astype(float),
                "worklife":     rng.integers(1, 4, len(r2)),
                "attrition":    pd.to_numeric(r2.get("Termd"), errors="coerce").fillna(0).astype(int),
            })
            a1 = pd.concat([a1, r2_map], ignore_index=True).head(n + len(r2_map))
            print(f"  ✅ Augmented Agent 1 with {len(r2_map)} rows from HRDataset_v14")
        except Exception as e:
            print(f"  ⚠️  HRDataset_v14 augmentation skipped ({e})")

    # ── Agent 2: Superstore & Store Performance (2 datasets per Section 7.1) ──
    raw_s1 = kaggle_download("vivek465/superstore-dataset-final", "Sample - Superstore.csv")
    raw_s2 = kaggle_download("kyanyoga/sample-store-data", "store_data.csv")
    n2 = n
    if raw_s1 is not None and "Sales" in raw_s1.columns:
        sales_vals = raw_s1["Sales"].dropna().astype(float).values
    else:
        sales_vals = np.array([])
    if raw_s2 is not None:
        sales_col = next((c for c in raw_s2.columns if "sales" in c.lower()), None)
        if sales_col:
            sales_vals = np.concatenate([sales_vals, raw_s2[sales_col].dropna().astype(float).values])
    if len(sales_vals) == 0:
        sales_vals = rng.uniform(90000, 350000, n2)
    elif len(sales_vals) < n2:
        sales_vals = np.pad(sales_vals, (0, n2 - len(sales_vals)), mode="wrap")
    sales_vals = sales_vals[:n2]

    a2 = pd.DataFrame({
        "sales":     sales_vals,
        "costs":     sales_vals * rng.uniform(0.55, 0.93, n2),
        "headcount": rng.integers(10, 45, n2),
        "orders":    rng.integers(200, 900, n2),
        "footfall":  rng.integers(800, 4000, n2),
        "rating":    rng.uniform(3.0, 5.0, n2),
    })
    a2["margin"] = (a2["sales"] - a2["costs"]) / a2["sales"]

    # ── Agent 3: Inventory & Item Demand (2 datasets per Section 7.1) ──────
    raw_inv1 = kaggle_download("pratyushraj1/retail-inventory-management-dataset", "inventory.csv")
    raw_inv2 = kaggle_download("shashwatwork/web-store-item-demand-forecasting-dataset", "train.csv")
    n3 = n
    if raw_inv1 is not None and "demand" in raw_inv1.columns:
        dem_vals = raw_inv1["demand"].dropna().astype(float).values
    else:
        dem_vals = np.array([])
    if raw_inv2 is not None:
        demand_col = next((c for c in raw_inv2.columns if c.lower() in ("sales", "demand", "units")), None)
        if demand_col:
            dem_vals = np.concatenate([dem_vals, raw_inv2[demand_col].dropna().astype(float).values])
    if len(dem_vals) == 0:
        dem_vals = rng.integers(80, 550, n3).astype(float)
    elif len(dem_vals) < n3:
        dem_vals = np.pad(dem_vals, (0, n3 - len(dem_vals)), mode="wrap")
    dem_vals = dem_vals[:n3]

    a3 = pd.DataFrame({
        "demand":    dem_vals,
        "stock":     rng.integers(50, 700, n3),
        "lead_time": rng.integers(1, 9, n3),
        "weather":   rng.uniform(-0.30, 0.35, n3),
        "promo":     rng.choice([0, 1], n3, p=[0.75, 0.25]),
    })
    a3["adj_demand"] = a3["demand"] * (1 + a3["weather"]) * (1 + a3["promo"] * 0.18) + rng.normal(0, 18, n3)

    print("\n  💾 Storing merged records …")
    with get_conn() as conn:
        conn.execute("DELETE FROM merged_datasets")
        for i in range(min(900, len(a1))):
            conn.execute(
                "INSERT INTO merged_datasets (agent_target,dataset_source,outlet_id,"
                "employee_age,overtime_hours,job_satisfaction,attrition_target,"
                "monthly_sales_usd,operating_cost_usd,tier_cluster_label,"
                "sku_demand,weather_impact_factor,stockout_target) VALUES (?,?,?,?,?,?,?,?,?,?,?,?,?)",
                ("All Agents", "IBM_HR+Superstore+Inventory",
                 f"OUT-{101+(i%10)}",
                 int(a1["age"].iloc[i]), float(a1["overtime"].iloc[i]),
                 int(a1["satisfaction"].iloc[i]), int(a1["attrition"].iloc[i]),
                 float(a2["sales"].iloc[i]), float(a2["costs"].iloc[i]), 0,
                 int(a3["demand"].iloc[i]), float(a3["weather"].iloc[i]),
                 int(a3["adj_demand"].iloc[i])))
        conn.commit()
    print("  ✅ Done.\n")
    return a1, a2, a3


def tier_the_10_outlets():
    """KMeans(k=4) on the 10 seeded outlets → Excellent/Good/Needs Attention/Critical (Section 7)."""
    with get_conn() as conn:
        outlets = pd.read_sql("SELECT outlet_id, monthly_revenue, avg_daily_orders "
                              "FROM outlets", conn)
    if outlets.empty:
        print("  ⚠️  No outlets seeded yet — skipping tiering.")
        return None

    # Section 7: "cluster the 10 seeded outlets by average daily revenue and order count"
    outlets["avg_daily_revenue"] = outlets["monthly_revenue"] / 30.0
    X = outlets[["avg_daily_revenue", "avg_daily_orders"]].fillna(0)
    k = min(4, outlets["outlet_id"].nunique())
    km = KMeans(n_clusters=k, random_state=42, n_init=15)
    labels = km.fit_predict(X)
    sil = float(silhouette_score(X, labels)) if k > 1 else 0.0
    print(f"  🔬 Agent2_KMeans_Tiering — k={k}: silhouette={sil:.4f}")
    save_ml_metrics("Agent2_KMeans_Tiering", f"KMeans(k={k})", sil, 0.0, 0.0, len(outlets), KMEANS_MODEL_PATH)
    joblib.dump(km, KMEANS_MODEL_PATH)

    # rank clusters by mean daily revenue + orders (normalized) descending → Excellent..Critical
    outlets["cluster"] = labels
    rev_norm = outlets.groupby("cluster")["avg_daily_revenue"].mean()
    ord_norm = outlets.groupby("cluster")["avg_daily_orders"].mean()
    cluster_rank = ((rev_norm / rev_norm.max()) + (ord_norm / ord_norm.max())).sort_values(ascending=False)
    cluster_to_tier = {cl: TIER_ORDER[i] for i, cl in enumerate(cluster_rank.index)}
    outlets["tier"] = outlets["cluster"].map(cluster_to_tier)

    with get_conn() as conn:
        for _, row in outlets.iterrows():
            conn.execute("UPDATE outlets SET tier_cluster=? WHERE outlet_id=?",
                         (row["tier"], row["outlet_id"]))
        conn.commit()
    print(f"  ✅ Outlet tiers assigned: {outlets['tier'].value_counts().to_dict()}")
    return km


def train_all_agents():
    print("=" * 60)
    print("  🚀 FranchiseOps AI — Multi-Algorithm Training Pipeline (Milestone 2)")
    print("=" * 60)
    a1, a2, a3 = generate_datasets()

    # ── Agent 1: Attrition Classification (7 algorithms) ──────────────────
    X1 = a1[["age", "satisfaction", "overtime", "tenure_yrs", "income", "worklife"]]
    y1 = a1["attrition"]
    X1tr, X1te, y1tr, y1te = train_test_split(X1, y1, test_size=0.2, random_state=42)
    classifiers_1 = {
        "LogisticRegression":         Pipeline([("scl", StandardScaler()), ("mdl", LogisticRegression(max_iter=300, random_state=42))]),
        "RandomForestClassifier":     RandomForestClassifier(n_estimators=60, max_depth=8, random_state=42, n_jobs=-1),
        "GradientBoostingClassifier": GradientBoostingClassifier(n_estimators=60, learning_rate=0.1, max_depth=3, random_state=42),
        "SVC_RBF":                    Pipeline([("scl", StandardScaler()), ("mdl", SVC(kernel="rbf", probability=True, random_state=42))]),
        "DecisionTreeClassifier":     DecisionTreeClassifier(max_depth=6, random_state=42),
        "AdaBoostClassifier":         AdaBoostClassifier(n_estimators=60, random_state=42),
        "KNeighborsClassifier":       Pipeline([("scl", StandardScaler()), ("mdl", KNeighborsClassifier(n_neighbors=15))]),
    }
    m1, bn1, auc1 = compare_classifiers(classifiers_1, X1tr, X1te, y1tr, y1te,
                                         "Agent1_Attrition", AGENT1_MODEL_PATH)
    print(f"  → ROC-AUC (attrition champion): {auc1:.4f}")

    # ── Agent 2: Revenue Regression (7 algorithms) ─────────────────────────
    X2r = a2[["costs", "headcount", "footfall", "rating"]]
    y2r = a2["sales"]
    X2rtr, X2rte, y2rtr, y2rte = train_test_split(X2r, y2r, test_size=0.2, random_state=42)
    regressors_2 = {
        "RandomForestRegressor":     RandomForestRegressor(n_estimators=60, max_depth=10, random_state=42, n_jobs=-1),
        "GradientBoostingRegressor": GradientBoostingRegressor(n_estimators=60, learning_rate=0.1, max_depth=4, random_state=42),
        "ExtraTreesRegressor":       ExtraTreesRegressor(n_estimators=60, max_depth=10, random_state=42, n_jobs=-1),
        "Ridge":                     Pipeline([("scl", StandardScaler()), ("mdl", Ridge(alpha=1.0))]),
        "DecisionTreeRegressor":     DecisionTreeRegressor(max_depth=8, random_state=42),
        "AdaBoostRegressor":         AdaBoostRegressor(n_estimators=60, random_state=42),
        "KNeighborsRegressor":       Pipeline([("scl", StandardScaler()), ("mdl", KNeighborsRegressor(n_neighbors=10))]),
    }
    m2r, bn2r, r2_2 = compare_regressors(regressors_2, X2rtr, X2rte, y2rtr, y2rte,
                                          "Agent2_Revenue", AGENT2_REG_PATH)

    # ── Agent 2 (continued): KMeans 4-tier outlet tiering ──────────────────
    tier_the_10_outlets()

    # ── Agent 3: Inventory Demand Regression (7 algorithms) ────────────────
    X3 = a3[["demand", "stock", "lead_time", "weather", "promo"]]
    y3 = a3["adj_demand"]
    X3tr, X3te, y3tr, y3te = train_test_split(X3, y3, test_size=0.2, random_state=42)
    regressors_3 = {
        "GradientBoostingRegressor": GradientBoostingRegressor(n_estimators=60, learning_rate=0.1, max_depth=4, random_state=42),
        "RandomForestRegressor":     RandomForestRegressor(n_estimators=60, max_depth=10, random_state=42, n_jobs=-1),
        "ExtraTreesRegressor":       ExtraTreesRegressor(n_estimators=60, max_depth=10, random_state=42, n_jobs=-1),
        "Ridge":                     Pipeline([("scl", StandardScaler()), ("mdl", Ridge(alpha=1.0))]),
        "DecisionTreeRegressor":     DecisionTreeRegressor(max_depth=8, random_state=42),
        "AdaBoostRegressor":         AdaBoostRegressor(n_estimators=60, random_state=42),
        "KNeighborsRegressor":       Pipeline([("scl", StandardScaler()), ("mdl", KNeighborsRegressor(n_neighbors=10))]),
    }
    m3, bn3, r2_3 = compare_regressors(regressors_3, X3tr, X3te, y3tr, y3te,
                                        "Agent3_Inventory", AGENT3_MODEL_PATH)

    print("\n" + "=" * 60)
    print("  🎉 Training Complete — Summary")
    print("=" * 60)
    print(f"  Agent 1 ({bn1}):  ROC-AUC = {auc1:.4f}   [{len(classifiers_1)} algorithms compared]")
    print(f"  Agent 2 ({bn2r}): R²      = {r2_2:.4f}   [{len(regressors_2)} algorithms compared] + KMeans 4-tier")
    print(f"  Agent 3 ({bn3}):  R²      = {r2_3:.4f}   [{len(regressors_3)} algorithms compared]")
    print(f"  Models saved to: {MODELS_DIR}")
    print("=" * 60)


if __name__ == "__main__":
    train_all_agents()


Overwriting train_m2_franchise.py


## Step 6b — Write Main Application (`app.py`)


In [31]:
%%writefile app.py
"""
app.py — FranchiseOps AI v4 FINAL (Modular Fast Engine)
Lean orchestrator — heavy tab logic lives in agent2_franchise.py, agent3_franchise.py, admin_dash.py
"""
import os, json, joblib, subprocess, numpy as np, pandas as pd
import streamlit as st
from streamlit_option_menu import option_menu
from config import AGENT1_MODEL_PATH, AGENT2_MODEL_PATH, AGENT2_REG_PATH, AGENT3_MODEL_PATH
from ui_theme import apply_theme, render_header, render_card, COLORS
from auth import render_auth_portal
from db import get_conn, load_chat_history, save_chat_message
from weather_context import get_city_weather
from notifications import send_alert, get_recent_alerts
from llm_engine_franchise import (orchestrate_3_agents_query, generate_debate_and_synthesis,
                        warmup_llm, is_llm_loaded, start_background_warmup, synthesize_erp_action)
from agent2_franchise import render_agent2_franchise
from agent3_franchise import render_agent3_franchise
from admin_dash import render_admin_dashboard

st.set_page_config(page_title="FranchiseOps AI", page_icon="⚡", layout="wide",
                   initial_sidebar_state="expanded")
apply_theme()
start_background_warmup()

if not st.session_state.get("token"):
    render_auth_portal(); st.stop()

username  = st.session_state.get("username", "guest")
user_role = st.session_state.get("role", "Franchise Owner")
is_admin  = user_role.lower() == "admin"

with st.sidebar:
    st.markdown(f'<div style="text-align:center;padding:10px 0;font-weight:700;font-size:18px;'
                f'color:{COLORS["text_heading"]};">⚡ FranchiseOps AI</div>', unsafe_allow_html=True)
    st.markdown(f'<div style="text-align:center;font-size:13px;color:{COLORS["text_muted"]};'
                f'margin-bottom:12px;">User: <b>{username}</b><br>'
                f'<span style="color:#0066cc;font-weight:600;">[{user_role}]</span></div>',
                unsafe_allow_html=True)
    tabs  = ["🤖 AI Copilot", "👥 Agent 1: Workforce", "🏬 Agent 2: Outlets",
             "📦 Agent 3: Inventory", "📊 Analytics & Retrain"]
    icons = ["chat-dots-fill", "people-fill", "building", "box-seam-fill", "bar-chart-fill"]
    if is_admin:
        tabs.append("🛡️ Admin Dashboard"); icons.append("shield-lock-fill")
    tabs.append("🚪 Sign Out"); icons.append("box-arrow-right")
    selected_tab = option_menu(menu_title=None, options=tabs, icons=icons, default_index=0,
        styles={
            "container": {"padding": "0!important", "background-color": "transparent"},
            "nav-link": {"font-size": "13px", "text-align": "left", "margin": "3px 0",
                         "border-radius": "10px", "color": COLORS["text_main"], "font-weight": "600"},
            "nav-link-selected": {"background-color": COLORS["accent"], "color": COLORS["accent_text"],
                                  "border": f"2px solid {COLORS['border']}"},
        })

if selected_tab == "🚪 Sign Out":
    st.session_state["token"] = None; st.rerun()

render_header("FranchiseOps AI", f"Module: {selected_tab}")

b1, b2 = st.columns([4, 1.2])
with b1:
    if is_llm_loaded():
        st.markdown('<div style="background:#d1fae5;border:2px solid #34d399;border-radius:10px;'
                    'padding:8px 16px;font-weight:600;color:#065f46;font-size:13px;">'
                    '⚡ <b>LLM GPU Engine:</b> Active on Tesla T4 (Qwen-2.5-3B Ready)</div>',
                    unsafe_allow_html=True)
    else:
        st.markdown('<div style="background:#bae8e8;border:2px solid #272343;border-radius:10px;'
                    'padding:8px 16px;font-weight:600;color:#272343;font-size:13px;">'
                    '⚡ <b>LLM GPU Engine:</b> Standby — warm up before use</div>',
                    unsafe_allow_html=True)
with b2:
    if not is_llm_loaded():
        if st.button("⚡ Warm Up LLM", key="warmup_btn", use_container_width=True):
            with st.spinner("Loading Qwen-2.5-3B from Drive cache..."):
                warmup_llm()
            st.rerun()


@st.cache_resource
def load_agents():
    if not os.path.exists(AGENT1_MODEL_PATH) or not os.path.exists(AGENT2_MODEL_PATH) or not os.path.exists(AGENT2_REG_PATH) or not os.path.exists(AGENT3_MODEL_PATH):
        try:
            from train_m2_franchise import train_all_agents
            train_all_agents()
        except Exception as e:
            print(f"Auto-training note: {e}")
    m1  = joblib.load(AGENT1_MODEL_PATH) if os.path.exists(AGENT1_MODEL_PATH) else None
    m2c = joblib.load(AGENT2_MODEL_PATH) if os.path.exists(AGENT2_MODEL_PATH) else None
    m2r = joblib.load(AGENT2_REG_PATH)   if os.path.exists(AGENT2_REG_PATH)   else None
    m3  = joblib.load(AGENT3_MODEL_PATH) if os.path.exists(AGENT3_MODEL_PATH) else None
    return m1, m2c, m2r, m3

agent1_m, agent2_c, agent2_r, agent3_m = load_agents()


def confidence_band(model, X_row):
    if model is None:
        return 0.5, 0.42, 0.58
    if hasattr(model, "predict_proba"):
        prob = float(model.predict_proba([X_row])[0][1])
    else:
        prob = float(np.clip(model.predict([X_row])[0], 0, 1))
    z, n = 1.96, 300
    lo = max(0.0, (prob+z**2/(2*n)-z*((prob*(1-prob)+z**2/(4*n))/n)**0.5)/(1+z**2/n))
    hi = min(1.0, (prob+z**2/(2*n)+z*((prob*(1-prob)+z**2/(4*n))/n)**0.5)/(1+z**2/n))
    return prob, lo, hi


with get_conn() as conn:
    n_out  = conn.execute("SELECT count(*) FROM outlets").fetchone()[0]
    n_st   = conn.execute("SELECT count(*) FROM staff").fetchone()[0]
    n_inv  = conn.execute("SELECT count(*) FROM inventory_records").fetchone()[0]
    n_alrt = conn.execute("SELECT count(*) FROM notifications").fetchone()[0]

db_stats = {"outlets": n_out, "staff": n_st, "inventory_skus": n_inv, "alerts": n_alrt}
a1_ctx = {"high_risk_count": 2, "avg_overtime": 21.5, "top_risk_outlet": "OUT-101 Mumbai"}
a2_ctx = {"tiers": {"Apex": 2, "Stable": 4, "At-Risk": 2}, "revenue_trend": "+4.2%"}
a3_ctx = {"critical_skus": ["Coffee Beans", "Eco Cups"], "reorder_urgency": "Immediate"}

# ─────────────────────────────────────────────────────────────────────────────
# TAB: AI COPILOT
# ─────────────────────────────────────────────────────────────────────────────
if selected_tab == "🤖 AI Copilot":
    render_card('<h3 style="margin:0 0 6px;">💬 Unified AI Copilot — Total Franchise Intelligence</h3>'
                '<p style="margin:0;color:#64748b;font-size:13px;">Powered by Qwen-2.5-3B on T4. '
                'All answers use live DB stats, city weather, attrition scores & inventory data.</p>')

    if "copilot_history" not in st.session_state:
        hist = load_chat_history(username, get_conn)
        if not hist:
            msg = "Welcome to FranchiseOps AI Copilot! Ask about outlet performance, staff attrition, or inventory risk."
            save_chat_message(username, "assistant", msg, get_conn)
            hist = [{"role": "assistant", "content": msg}]
        st.session_state["copilot_history"] = hist

    for m in st.session_state["copilot_history"]:
        bg    = "#e3f6f5" if m["role"] == "user" else "white"
        label = "🧑 You" if m["role"] == "user" else "⚡ Copilot"
        st.markdown(f'<div class="pn-card" style="background:{bg};border-left:5px solid '
                    f'{COLORS["accent"] if m["role"]=="user" else COLORS["border"]};">'
                    f'<b>{label}:</b><br>{m["content"]}</div>', unsafe_allow_html=True)

    inp_col, clr_col = st.columns([8, 1])
    with inp_col:
        with st.form("copilot_form", clear_on_submit=True):
            user_q = st.text_input("", placeholder="e.g. 'Why is OUT-101 Mumbai struggling with staff attrition?'")
            fa, fb, fc = st.columns([3, 1, 1.4])
            with fa: submit = st.form_submit_button("🚀 Ask Copilot")
            with fb: debate = st.form_submit_button("🔍 Debate View")
            with fc: erp_action = st.form_submit_button("🗂️ Generate ERP Action")
    with clr_col:
        if st.button("🗑️", help="Clear history"):
            from db import clear_chat_history
            clear_chat_history(username, get_conn)
            st.session_state["copilot_history"] = []; st.rerun()

    if erp_action and user_q.strip():
        save_chat_message(username, "user", user_q, get_conn)
        st.session_state["copilot_history"].append({"role": "user", "content": user_q})
        with st.spinner("⚡ Synthesizing 3-agent ERP action..."):
            erp_json = synthesize_erp_action(user_q, a1_ctx, a2_ctx, a3_ctx, db_stats)
        st.session_state["last_erp_action"] = erp_json  # persist across reruns, don't just flash it
        ans = f"**ERP Action Generated:** {erp_json.get('action_type', 'N/A')} — {erp_json.get('recommended_step', '')}"
        save_chat_message(username, "assistant", ans, get_conn)
        st.session_state["copilot_history"].append({"role": "assistant", "content": ans})

    if st.session_state.get("last_erp_action"):
        render_card('<h4 style="margin:0 0 8px;">🗂️ Structured ERP Action (Phase 3 output)</h4>')
        st.json(st.session_state["last_erp_action"])
        if st.button("✖️ Clear ERP Action", key="btn_clear_erp"):
            st.session_state["last_erp_action"] = None
            st.rerun()

    if (submit or debate) and user_q.strip():
        save_chat_message(username, "user", user_q, get_conn)
        st.session_state["copilot_history"].append({"role": "user", "content": user_q})
        if debate:
            with st.spinner("⚡ Single-pass debate (~2 sec)..."):
                res = generate_debate_and_synthesis(user_q, a1_ctx, a2_ctx, a3_ctx, db_stats)
            dc1, dc2, dc3 = st.columns(3)
            for col, key, label, color in [
                (dc1, "agent1", "Workforce Retention", COLORS["accent"]),
                (dc2, "agent2", "Outlet Clustering",   "#34d399"),
                (dc3, "agent3", "Inventory & Weather", "#f87171"),
            ]:
                col.markdown(f'<div class="pn-card" style="border-top:4px solid {color};">'
                             f'<span class="agent-badge">{label}</span><br><br>{res[key]}</div>',
                             unsafe_allow_html=True)
            ans = f"**Executive Synthesis:** {res['synthesis']}"
        else:
            with st.spinner("⚡ Generating answer (~1.5 sec)..."):
                ans = orchestrate_3_agents_query(user_q, a1_ctx, a2_ctx, a3_ctx, db_stats)
        save_chat_message(username, "assistant", ans, get_conn)
        st.session_state["copilot_history"].append({"role": "assistant", "content": ans})
        st.rerun()

# ─────────────────────────────────────────────────────────────────────────────
# ─────────────────────────────────────────────────────────────────────────────
# TAB: AGENT 1 — WORKFORCE
# ─────────────────────────────────────────────────────────────────────────────
elif selected_tab == "👥 Agent 1: Workforce":
    render_card('<h3 style="margin:0;">👥 Agent 1: Staff Attrition Risk Predictor</h3>')
    with get_conn() as conn:
        staff_df = pd.read_sql("SELECT * FROM staff", conn)

    def _s_int(val, default):
        return default if (val is None or pd.isna(val)) else int(val)
    def _s_float(val, default):
        return default if (val is None or pd.isna(val)) else float(val)

    c1, c2 = st.columns(2)
    with c1:
        sel  = st.selectbox("Staff Member", staff_df["employee_name"].tolist())
        row  = staff_df[staff_df["employee_name"] == sel].iloc[0]
        sim_ot  = st.slider("Simulate Overtime Hrs", 0.0, 35.0, _s_float(row.get("weekly_overtime_hrs"), 18.0))
        sim_sat = st.slider("Simulate Job Satisfaction", 1, 5, _s_int(row.get("job_satisfaction"), 3))
    with c2:
        sim_age    = _s_int(row.get("employee_age"), 30)
        sim_tenure = _s_float(row.get("tenure_years"), 4.0)
        sim_income = _s_float(row.get("monthly_salary"), 55000.0)
        sim_wl     = _s_int(row.get("work_life_balance"), 3)
        X_row = [sim_age, sim_sat, sim_ot, sim_tenure, sim_income, sim_wl]
        prob, lo, hi = confidence_band(agent1_m, X_row)
        badge_c = "#f87171" if prob > 0.6 else ("#ffd803" if prob > 0.35 else "#34d399")
        st.markdown(
            f'<div style="background:{badge_c};padding:16px;border-radius:12px;'
            f'border:2px solid {COLORS["border"]};">'
            f'<span class="agent-badge">Agent 1</span>'
            f'<h2 style="color:#272343;margin:8px 0 0;">{prob*100:.1f}% Attrition Risk</h2>'
            f'<p style="font-weight:600;margin:4px 0;">95% CI: {lo*100:.1f}% — {hi*100:.1f}%</p>'
            f'</div>', unsafe_allow_html=True)
        from llm_engine_franchise import generate_json
        if st.button("✨ AI Retention Strategy"):
            with st.spinner("Generating (~2 sec)..."):
                s = generate_json(
                    f"{sel}: {sim_ot}h overtime, satisfaction {sim_sat}/5, salary ₹{sim_income:,.0f}.",
                    ["retention_action", "bonus_recommendation", "priority_level"])
            st.json(s)

# ─────────────────────────────────────────────────────────────────────────────
# TAB: AGENT 2 — OUTLETS (modular)
# ─────────────────────────────────────────────────────────────────────────────
elif selected_tab == "🏬 Agent 2: Outlets":
    render_agent2_franchise(agent2_c, agent2_r, username, db_stats, a1_ctx, a3_ctx,
                            send_alert, confidence_band)

# ─────────────────────────────────────────────────────────────────────────────
# TAB: AGENT 3 — INVENTORY (modular)
# ─────────────────────────────────────────────────────────────────────────────
elif selected_tab == "📦 Agent 3: Inventory":
    render_agent3_franchise(agent3_m, username, db_stats, a1_ctx, a2_ctx, send_alert)

# ─────────────────────────────────────────────────────────────────────────────
# TAB: ANALYTICS & RETRAIN
# ─────────────────────────────────────────────────────────────────────────────
elif selected_tab == "📊 Analytics & Retrain":
    render_card('<h3 style="margin:0;">📊 Enterprise Analytics & Model Management</h3>')
    kc = st.columns(4)
    for col, icon, label, val in [
        (kc[0], "🏬", "Outlets",   n_out),
        (kc[1], "👥", "Staff",     n_st),
        (kc[2], "📦", "SKUs",      n_inv),
        (kc[3], "🔔", "Alerts",    n_alrt),
    ]:
        col.markdown(f'<div class="pn-card" style="text-align:center;padding:14px;">'
                     f'<div style="font-size:26px;">{icon}</div>'
                     f'<h2 style="margin:4px 0;">{val}</h2>'
                     f'<p style="margin:0;color:{COLORS["text_muted"]};font-size:12px;">{label}</p>'
                     f'</div>', unsafe_allow_html=True)
    st.markdown("---")
    mc1, mc2 = st.columns([1, 1.5])
    with mc1:
        render_card('<h4 style="margin:0 0 8px;">🔄 1-Click Retrain</h4>')
        if st.button("🔄 Retrain All Agents Now"):
            with st.spinner("Training... (~2-3 min)"):
                res = subprocess.run(["python", "train_m2_franchise.py"], capture_output=True, text=True, timeout=300)
            load_agents.clear()
            (st.success if res.returncode == 0 else st.error)(
                "✅ All agents retrained!" if res.returncode == 0 else "❌ Training failed.")
            st.code((res.stdout if res.returncode == 0 else res.stderr)[-1000:])
    with mc2:
        with get_conn() as conn:
            try:
                ml_df = pd.read_sql("SELECT agent_name,model_name,r2_score,accuracy,"
                                    "training_rows,created_at FROM ml_models ORDER BY id DESC", conn)
                st.dataframe(ml_df, use_container_width=True, hide_index=True)
            except Exception:
                st.info("No model history yet.")
    st.markdown("---")
    render_card('<h4 style="margin:0 0 8px;">🔔 Recent Alerts</h4>')
    for a in get_recent_alerts(10):
        st.markdown(f'<div style="border-bottom:1px solid #bae8e8;padding:5px 0;font-size:13px;">'
                    f'<b>[{a[1].upper()}]</b> {a[3]} '
                    f'<span style="color:{COLORS["text_muted"]};float:right;">{a[4]}</span></div>',
                    unsafe_allow_html=True)

# ─────────────────────────────────────────────────────────────────────────────
# TAB: ADMIN DASHBOARD (modular)
# ─────────────────────────────────────────────────────────────────────────────
elif selected_tab == "🛡️ Admin Dashboard":
    if not is_admin:
        st.error("🔒 Admin access required.")
    else:
        render_admin_dashboard(project="franchise")


Overwriting app.py


## Step 7 — Launch Streamlit App via ngrok


In [32]:
import subprocess, time, os
from pyngrok import ngrok
try:
    from config import NGROK_AUTHTOKEN as NGROK_AUTH_TOKEN
except ImportError:
    from config import NGROK_AUTH_TOKEN

if NGROK_AUTH_TOKEN:
    ngrok.set_auth_token(NGROK_AUTH_TOKEN)
    public_url = ngrok.connect(8501).public_url
    print("🚀 App Published at:", public_url)
else:
    print("Running locally on port 8501.")

process = subprocess.Popen(["streamlit", "run", "app.py",
                            "--server.port=8501", "--server.headless=true"])
print("✅ Streamlit started (PID:", process.pid, ")")


🚀 App Published at: https://wincing-stegosaur-caboose.ngrok-free.dev
✅ Streamlit started (PID: 2374 )


## Step 8 — Stop Application & Free GPU Memory


In [ ]:
try:
    process.terminate()
    ngrok.kill()
    print("🛑 Streamlit and ngrok terminated successfully.")
except Exception as e:
    print("Info:", e)
